# ⚙️ Sistemas de Produção
### Inteligência Artificial — Ciência da Computação · Aula 07

<a href="https://colab.research.google.com/github/profluiscaparroz/inteligencia-artificial/blob/main/07-sistemas-producao/07_Sistemas_Producao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

**Objetivo desta aula:** Compreender os fundamentos dos Sistemas de Produção, sua arquitetura clássica, mecanismos de inferência e aplicações práticas em resolução de problemas e representação de conhecimento.

---

### 🎯 Objetivos de aprendizagem

Ao final desta aula, você será capaz de:

1. **Explicar** o que é um sistema de produção e seus três componentes (regras, memória de trabalho e motor de inferência);
2. **Executar à mão** o ciclo **reconhecer–agir** (*match → select → execute*);
3. **Implementar** encadeamento **progressivo** (*forward chaining*) e **regressivo** (*backward chaining*) em Python;
4. **Comparar** as estratégias de resolução de conflitos e prever como elas mudam o comportamento do sistema;
5. **Entender** por que a **refratariedade** evita *loops* e por que o algoritmo **RETE** acelera o casamento de padrões;
6. **Projetar** um sistema de produção para um problema real (casa inteligente, diagnóstico, personagem de jogo).

### 🗺️ Roteiro da aula

| Parte | Conteúdo | Destaques práticos |
|:-:|---|---|
| 0 | Preparação do ambiente | — |
| 1 | O que são sistemas de produção, história e características | Exemplo mínimo (termostato) e Sistemas de Post 🎬 |
| 2 | Arquitetura e ciclo reconhecer–agir | — |
| 3 | Implementação com *forward chaining* | Animação do ciclo 🎬, mesmo conjunto de regras com fatos diferentes |
| 4 | *Backward chaining* | Animação da árvore de prova com *backtracking* 🎬 |
| 5 | Comparação *forward* × *backward* | Experimento com 240 regras |
| 6 | Tópicos avançados | RETE (experimento), estratégias de conflito 🎛️, refratariedade e *loops* |
| 7 | Aplicações reais | NPC de jogo controlado por regras 🎬 |
| 8–10 | Vantagens, exercícios e referências | — |

> 💡 **Como usar:** execute as células **em ordem** (`Shift + Enter`). 🎬 = animação (use o botão ▶), 🎛️ = controle interativo. Depois de **cada resultado** há uma seção **🔎 Como interpretar**, que explica o que aconteceu, por que acontece e como ler a saída.

---

## Parte 0 — Preparação do ambiente

No Google Colab tudo já vem instalado. Execute a célula abaixo antes das demais.

In [ ]:
# =====================================================
# IMPORTAÇÕES E FUNÇÕES AUXILIARES — execute primeiro
# =====================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

# Animações renderizadas como player HTML/JavaScript (funciona no Colab sem ffmpeg)
plt.rcParams['animation.html'] = 'jshtml'
plt.rcParams['animation.embed_limit'] = 60
plt.rcParams['figure.dpi'] = 90

CORES = {'azul': '#1f77b4', 'vermelho': '#d62728', 'verde': '#2ca02c', 'laranja': '#ff7f0e',
         'roxo': '#9467bd', 'cinza': '#bbbbbb', 'amarelo': '#ffd54f'}


def mostrar_animacao(anim):
    # Exibe uma animação do matplotlib como player interativo (▶ ⏸ ⏭) e fecha a figura estática.
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


print('✅ Ambiente pronto!')

#### 🔎 Como interpretar esta célula

- Carregamos `numpy` e `matplotlib` (gráficos e animações) e `ipywidgets` (controles interativos).
- A função `mostrar_animacao(anim)` transforma uma animação do matplotlib em um **player** com botões ▶, ⏸ e avanço quadro a quadro. Use o botão de avançar quadro (⏭) para estudar **cada passo** com calma.
- Se aparecer `✅ Ambiente pronto!`, pode seguir.

## 1. O que são Sistemas de Produção?

**Sistemas de Produção** (Production Systems) são uma das arquiteturas mais fundamentais e influentes da Inteligência Artificial, propostos inicialmente por **Allen Newell** e **Herbert Simon** na década de 1970.

Um Sistema de Produção é um **modelo computacional** que representa conhecimento na forma de **regras condição-ação** (também chamadas de **regras de produção**) e executa essas regras para resolver problemas, tomar decisões ou realizar inferências.

### 1.1 Definição Formal

Segundo **Nilsson** (1998), um Sistema de Produção consiste em:

1. **Base de Conhecimento (Knowledge Base):** Conjunto de regras de produção da forma:
   ```
   SE <condição> ENTÃO <ação>
   ```

2. **Base de Dados (Working Memory):** Fatos conhecidos sobre o estado atual do sistema

3. **Interpretador (Inference Engine):** Mecanismo que decide quais regras aplicar e em que ordem

A cada ciclo de execução, o sistema:
- **Reconhece (Match):** Identifica regras cujas condições são satisfeitas pelos fatos atuais
- **Seleciona (Select):** Escolhe uma regra entre as aplicáveis (resolução de conflitos)
- **Executa (Execute):** Aplica a ação da regra, potencialmente modificando a base de dados

Este ciclo é conhecido como **Ciclo Reconhecer-Agir** (Recognize-Act Cycle).

### 1.2 Breve História

A evolução dos Sistemas de Produção está intimamente ligada ao desenvolvimento da IA:

- **1943 — Post:** Emil Post introduziu **Sistemas de Post**, formalismo teórico para reescrita de strings usando regras de produção, precursor dos sistemas modernos.

- **1956 — Logic Theorist:** Newell, Shaw e Simon desenvolveram o **Logic Theorist**, primeiro programa a provar teoremas matemáticos usando raciocínio simbólico — demonstrado na Conferência de Dartmouth.

- **1959 — General Problem Solver (GPS):** Newell e Simon criaram o GPS, sistema universal para resolução de problemas baseado em análise meios-fins e regras de produção.

- **1972 — MYCIN:** Edward Shortliffe desenvolveu MYCIN, sistema especialista para diagnóstico de infecções sanguíneas, usando ~600 regras de produção com fatores de certeza.

- **1974 — Arquitetura ACT:** John Anderson propôs ACT (Adaptive Control of Thought), arquitetura cognitiva baseada em regras de produção para modelar cognição humana.

- **1979 — OPS5:** Forgy desenvolveu OPS5 (Official Production System 5), linguagem eficiente para sistemas de produção usando o **algoritmo RETE** para casamento de padrões.

- **1982 — RETE Algorithm:** Charles Forgy publica algoritmo RETE, revolucionando eficiência de sistemas baseados em regras ao evitar recomputação desnecessária.

- **Anos 1980 — Era dos Sistemas Especialistas:** Explosão de aplicações comerciais (R1/XCON para configuração de computadores, DENDRAL para química).

- **Anos 1990-2000 — Declínio e Evolução:** Limitações dos sistemas baseados em regras levaram a abordagens híbridas, combinando regras com aprendizado de máquina e raciocínio probabilístico.

- **Atualidade:** Sistemas de produção continuam relevantes em motores de regras de negócio (Drools, CLIPS), sistemas embarcados e arquiteturas cognitivas (SOAR, ACT-R).

### 1.3 Características Fundamentais

Sistemas de Produção possuem características distintivas:

#### 1.3.1 Modularidade
Cada regra é uma unidade independente de conhecimento. Adicionar, remover ou modificar regras não afeta necessariamente outras regras.

#### 1.3.2 Uniformidade
Todo conhecimento é representado no mesmo formalismo (regras IF-THEN), facilitando manipulação e interpretação.

#### 1.3.3 Naturalidade
Regras refletem forma natural de expressar conhecimento: "SE está chovendo ENTÃO leve guarda-chuva".

#### 1.3.4 Separação Conhecimento-Controle
**Conhecimento** (regras) é separado do **controle** (motor de inferência). Permite alterar estratégia de busca sem modificar regras.

#### 1.3.5 Transparência
Raciocínio é facilmente explicável — sequência de regras disparadas forma traço de execução compreensível.

#### 1.3.6 Opacidade Direcional
Ordem de execução não é explícita nas regras — depende do motor de inferência e estado da memória de trabalho.

### 1.4 O exemplo mais simples possível: um termostato inteligente

Antes das implementações completas, vamos ver um sistema de produção em **~20 linhas**. Repare nos três componentes:

| Componente | No código |
|---|---|
| **Memória de trabalho** | o dicionário `memoria` (temperatura, estado do ar, janela) |
| **Base de regras** | a lista `regras_termostato`: cada regra é um par **condição → ação** |
| **Motor de inferência** | o laço `while`: *match* (quem se aplica?), *select* (escolhe uma), *execute* (age) |

In [ ]:
# Um sistema de produção MÍNIMO: memória de trabalho = dicionário; regras = (nome, condição, ação)
memoria = {'temperatura': 31, 'ar_condicionado': 'desligado', 'janela': 'aberta'}

regras_termostato = [
    ('R1: fechar a janela antes de ligar o ar',
     lambda m: m['temperatura'] > 28 and m['janela'] == 'aberta',
     lambda m: m.update(janela='fechada')),
    ('R2: ligar o ar se está quente e a janela está fechada',
     lambda m: m['temperatura'] > 28 and m['janela'] == 'fechada' and m['ar_condicionado'] == 'desligado',
     lambda m: m.update(ar_condicionado='ligado')),
    ('R3: o ar ligado esfria o ambiente (−3 °C)',
     lambda m: m['ar_condicionado'] == 'ligado' and m['temperatura'] > 22,
     lambda m: m.update(temperatura=m['temperatura'] - 3)),
    ('R4: desligar o ar quando ficar agradável',
     lambda m: m['ar_condicionado'] == 'ligado' and m['temperatura'] <= 22,
     lambda m: m.update(ar_condicionado='desligado')),
]

print(f'Ciclo 0: {"(estado inicial)":<52} → {memoria}')
ciclo = 0
while True:
    conflito = [r for r in regras_termostato if r[1](memoria)]   # 1) MATCH: quais regras se aplicam?
    if not conflito:
        print('Nenhuma regra aplicável → o sistema PARA.')
        break
    nome, _, acao = conflito[0]                                    # 2) SELECT: escolhe a primeira
    acao(memoria)                                                  # 3) EXECUTE: altera a memória
    ciclo += 1
    print(f'Ciclo {ciclo}: {nome:<52} → {memoria}')

#### 🔎 Como interpretar o resultado

Cada linha é **um ciclo reconhecer–agir**. Acompanhe:

| Ciclo | O que aconteceu | Por quê |
|:-:|---|---|
| 1 | **R1** fecha a janela | Está quente (31 °C) e a janela está aberta. R1 é a **única** regra aplicável |
| 2 | **R2** liga o ar | Agora a janela está fechada, então a condição de R2 ficou verdadeira. **A ação de R1 "habilitou" R2** |
| 3–5 | **R3** dispara três vezes (31 → 28 → 25 → 22) | Enquanto o ar está ligado e a temperatura é > 22, R3 continua aplicável. Cada disparo **muda** a memória, então é uma situação nova |
| 6 | **R4** desliga o ar | A temperatura chegou a 22 °C |
| — | O sistema **para** | Nenhuma condição é verdadeira: não há mais nada a fazer |

- **Conceito-chave 1, encadeamento:** ninguém programou a sequência "fechar → ligar → esfriar → desligar". Ela **emergiu** da interação entre as regras e a memória: cada ação cria as condições para a próxima regra.
- **Conceito-chave 2, parada:** um sistema de produção termina quando o **conjunto de conflito** (regras aplicáveis) fica **vazio**, ou quando alguma regra executa uma ação de "parar".
- **💡 Experimente:** mude `temperatura` para 25 no início. Nenhuma regra se aplica e o sistema para imediatamente. Ou troque a ordem de R1 e R2 na lista: o resultado muda? Por quê?

### 1.5 As origens: Sistemas de Post (regras que reescrevem palavras)

Em 1943, **Emil Post** mostrou que regras do tipo **"troque o trecho X por Y"** são suficientes para expressar **qualquer computação** — elas são tão poderosas quanto uma máquina de Turing. É daí que vem o nome **regra de produção**: cada regra **produz** uma nova palavra a partir da anterior.

No exemplo abaixo, três regras ordenam as letras de uma palavra:

| Regra | Significado |
|---|---|
| `ba → ab` | um **b** antes de um **a** está fora de ordem: troque |
| `ca → ac` | idem para **c** antes de **a** |
| `cb → bc` | idem para **c** antes de **b** |

In [ ]:
# Sistema de Post / algoritmo de Markov: regras que REESCREVEM palavras.
# Em cada passo aplica-se a PRIMEIRA regra cujo lado esquerdo aparece na palavra (na 1ª ocorrência).
regras_post = [('ba', 'ab'), ('ca', 'ac'), ('cb', 'bc')]


def reescrever(palavra, regras, max_passos=50):
    passos = [(palavra, None, None)]
    for _ in range(max_passos):
        for esq, dir_ in regras:
            pos = palavra.find(esq)
            if pos >= 0:
                palavra = palavra[:pos] + dir_ + palavra[pos + len(esq):]
                passos.append((palavra, f'{esq} → {dir_}', pos))
                break
        else:          # nenhuma regra se aplica: chegamos à "forma normal"
            break
    return passos


passos_post = reescrever('cbacba', regras_post)
for i, (p, regra, pos) in enumerate(passos_post):
    info = f'aplicou {regra} na posição {pos}' if regra else 'palavra inicial'
    print(f'passo {i}: {p}   ({info})')

#### 🔎 Como interpretar o resultado

- A palavra `cbacba` foi transformada em `aabbcc` (letras **ordenadas**) em **9 passos**, usando só três regras de troca.
- **Como o motor escolheu cada passo?** Ele percorre as regras **na ordem** (`ba→ab`, depois `ca→ac`, depois `cb→bc`) e aplica a **primeira** que encontra na palavra. Isso é uma **estratégia de resolução de conflito** por ordem de definição. Por exemplo, no passo 3 a palavra `cabcab` continha `ca` **e** `cb`, mas como não havia `ba`, foi aplicada `ca→ac` (a segunda da lista).
- **Por que o processo termina?** Cada troca coloca uma letra "menor" à frente de uma "maior", e isso só pode acontecer um número finito de vezes. Quando a palavra está ordenada, **nenhum** lado esquerdo (`ba`, `ca`, `cb`) aparece mais: o conjunto de conflito fica **vazio**.
- **💡 Conceito-chave:** Post provou que esse formalismo simples, **"se aparece X, troque por Y"**, é **Turing-completo**: pode expressar qualquer algoritmo. As regras de produção dos sistemas especialistas e das gramáticas formais (Chomsky) descendem diretamente dessa ideia.

In [ ]:
# 🎬 ANIMAÇÃO — a palavra sendo reescrita, regra a regra
cor_letra = {'a': '#90caf9', 'b': '#a5d6a7', 'c': '#ffcc80'}
fig, ax = plt.subplots(figsize=(8, 2.6))

def quadro_post(k):
    ax.clear(); ax.axis('off')
    palavra, regra, pos = passos_post[k]
    for j, letra in enumerate(palavra):
        destaque = regra is not None and pos <= j < pos + 2
        ax.add_patch(plt.Rectangle((j, 0), 0.9, 0.9, color=cor_letra[letra],
                                   ec='red' if destaque else 'k', lw=3 if destaque else 1))
        ax.text(j + 0.45, 0.45, letra, ha='center', va='center', fontsize=22, fontweight='bold')
    ax.set_xlim(-0.2, len(palavra)); ax.set_ylim(-0.6, 1.4)
    titulo = f'Passo {k}: ' + (f'regra {regra} (contorno vermelho = trecho reescrito)' if regra else 'palavra inicial')
    if k == len(passos_post) - 1:
        titulo += '  →  nenhuma regra se aplica: FIM'
    ax.set_title(titulo, fontsize=11)

anim = animation.FuncAnimation(fig, quadro_post, frames=len(passos_post), interval=900)
mostrar_animacao(anim)

#### 🔎 Como interpretar a animação

- Cada caixa é uma letra (azul = **a**, verde = **b**, laranja = **c**). O **contorno vermelho** mostra o trecho que acabou de ser **reescrito** no passo atual.
- Observe as letras azuis "migrando" para a esquerda e as laranjas para a direita, **duas posições por vez**, como num *bubble sort* em que cada troca é uma regra.
- No último quadro o título indica **"nenhuma regra se aplica: FIM"**. É a **condição de parada** de todo sistema de produção.

## 2. Arquitetura de um Sistema de Produção

### 2.1 Componentes Principais

```
┌─────────────────────────────────────────────────┐
│          SISTEMA DE PRODUÇÃO                    │
├─────────────────────────────────────────────────┤
│                                                 │
│  ┌──────────────────────────────────────┐      │
│  │   BASE DE CONHECIMENTO               │      │
│  │                                      │      │
│  │  Regra 1: SE A E B ENTÃO C          │      │
│  │  Regra 2: SE C ENTÃO D              │      │
│  │  Regra 3: SE D E E ENTÃO F          │      │
│  │  ...                                │      │
│  └──────────────────────────────────────┘      │
│                    ↑                            │
│                    │                            │
│         ┌──────────┴─────────┐                 │
│         │  MOTOR DE INFERÊNCIA │                 │
│         │                      │                 │
│         │  1. Match (Casar)   │                 │
│         │  2. Select (Resolver) │                │
│         │  3. Execute (Agir)   │                │
│         └──────────┬─────────┘                 │
│                    │                            │
│                    ↓                            │
│  ┌──────────────────────────────────────┐      │
│  │   MEMÓRIA DE TRABALHO (Working Memory) │      │
│  │                                      │      │
│  │  Fato 1: A é verdadeiro             │      │
│  │  Fato 2: B é verdadeiro             │      │
│  │  Fato 3: temperatura = 25           │      │
│  │  ...                                │      │
│  └──────────────────────────────────────┘      │
│                                                 │
└─────────────────────────────────────────────────┘
```

### 2.2 Ciclo de Execução

O **Ciclo Reconhecer-Agir** se repete até que nenhuma regra seja aplicável ou objetivo seja alcançado:

**1. MATCH (Reconhecimento)**
- Motor de inferência compara condições de todas as regras com fatos na memória de trabalho
- Gera **conjunto de conflitos** (conflict set): regras cujas condições são satisfeitas

**2. SELECT (Resolução de Conflitos)**
- Escolhe uma regra do conjunto de conflitos usando estratégia de resolução:
  - **Especificidade:** Regras mais específicas têm prioridade
  - **Recência:** Regras que usam fatos mais recentes
  - **Ordenação explícita:** Prioridades definidas pelo projetista
  - **Refratariedade:** Evita disparar mesma regra com mesmos fatos repetidamente

**3. EXECUTE (Ação)**
- Executa ação da regra selecionada
- Modifica memória de trabalho (adiciona/remove fatos)
- Retorna ao passo 1

### 2.3 Estratégias de Controle

Duas estratégias principais de inferência:

#### Forward Chaining (Encadeamento para Frente)
- **Data-driven:** Parte dos fatos conhecidos e deriva conclusões
- **Bottom-up:** Das premissas para conclusões
- Uso: Sistemas reativos, monitoramento, alertas

```
Fatos: A, B → Aplica regra → Conclui C → Aplica regra → Conclui D
```

#### Backward Chaining (Encadeamento para Trás)
- **Goal-driven:** Parte de objetivo e busca fatos que o suportem
- **Top-down:** Da conclusão desejada para premissas
- Uso: Diagnóstico, sistemas especialistas consultivos

```
Objetivo: D → Precisa de C → Precisa de A e B → Verifica fatos
```

## 3. Implementação de Sistema de Produção Básico

In [ ]:
from typing import List, Dict, Callable, Any, Set
from dataclasses import dataclass
import re

@dataclass
class Regra:
    """
    Representa uma regra de produção.

    Atributos:
    ----------
    nome : str
        Identificador da regra
    condicoes : List[str]
        Lista de condições que devem ser satisfeitas
    acoes : List[str]
        Lista de ações a executar quando regra dispara
    prioridade : int
        Prioridade para resolução de conflitos (maior = mais prioritária)
    """
    nome: str
    condicoes: List[str]
    acoes: List[str]
    prioridade: int = 0

    def __repr__(self):
        conds = " E ".join(self.condicoes)
        acts = ", ".join(self.acoes)
        return f"Regra[{self.nome}]: SE {conds} ENTÃO {acts}"

class SistemaProducao:
    """
    Implementação de Sistema de Produção com Forward Chaining.

    Parâmetros:
    -----------
    estrategia_resolucao : str
        Estratégia para resolver conflitos: 'prioridade', 'especificidade', 'primeira'
    debug : bool
        Se True, exibe traço de execução detalhado
    """

    def __init__(self, estrategia_resolucao='prioridade', debug=True):
        self.base_conhecimento: List[Regra] = []
        self.memoria_trabalho: Set[str] = set()
        self.estrategia = estrategia_resolucao
        self.debug = debug
        self.historico_regras = []  # Rastreamento de execução
        self.regras_disparadas = set()  # Refratariedade: instanciações já disparadas
        self.timestamps: Dict[str, int] = {}  # Momento em que cada fato entrou na memória
        self.relogio = 0
        self.trace = []  # Registro de cada ciclo (usado nas animações)

    def adicionar_regra(self, nome: str, condicoes: List[str], 
                       acoes: List[str], prioridade: int = 0):
        """
        Adiciona regra à base de conhecimento.
        """
        regra = Regra(nome, condicoes, acoes, prioridade)
        self.base_conhecimento.append(regra)
        if self.debug:
            print(f"✓ Regra adicionada: {regra}")

    def adicionar_fato(self, fato: str):
        """
        Adiciona fato à memória de trabalho.
        """
        self._inserir(fato)
        if self.debug:
            print(f"✓ Fato adicionado: {fato}")

    def _inserir(self, fato: str) -> bool:
        """Insere fato na memória registrando o momento (timestamp). Retorna True se o fato é novo."""
        if fato in self.memoria_trabalho:
            return False
        self.relogio += 1
        self.memoria_trabalho.add(fato)
        self.timestamps[fato] = self.relogio
        return True

    def _instanciacao(self, regra: Regra):
        """
        Identifica UMA instanciação da regra: a regra + os fatos (com timestamps) que a satisfazem.
        A refratariedade proíbe disparar duas vezes a MESMA instanciação.
        """
        positivas = [c.strip() for c in regra.condicoes if not c.strip().startswith("NOT ")]
        return (regra.nome, tuple(sorted((c, self.timestamps.get(c, 0)) for c in positivas)))

    def _avaliar_condicao(self, condicao: str) -> bool:
        """
        Avalia se uma condição é satisfeita pela memória de trabalho.

        Suporta:
        - Fatos simples: "animal(cachorro)"
        - Negação: "NOT animal(gato)"
        """
        condicao = condicao.strip()

        # Verifica negação
        if condicao.startswith("NOT "):
            fato = condicao[4:].strip()
            return fato not in self.memoria_trabalho

        return condicao in self.memoria_trabalho

    def _match(self) -> List[Regra]:
        """
        Fase MATCH: Identifica regras aplicáveis.

        Retorna:
        --------
        Lista de regras cujas condições são todas satisfeitas.
        """
        conflito = []

        for regra in self.base_conhecimento:
            # Verifica se todas as condições são satisfeitas
            if not all(self._avaliar_condicao(cond) for cond in regra.condicoes):
                continue

            # Refratariedade: a mesma instanciação (regra + mesmos fatos) não dispara duas vezes
            if self._instanciacao(regra) in self.regras_disparadas:
                continue

            conflito.append(regra)

        return conflito

    def _select(self, conflito: List[Regra]) -> Regra:
        """
        Fase SELECT: Resolve conflito escolhendo uma regra.

        Estratégias:
        -----------
        - 'prioridade': Maior prioridade
        - 'especificidade': Mais condições (mais específica)
        - 'primeira': Primeira regra adicionada
        """
        if not conflito:
            return None

        if self.estrategia == 'prioridade':
            return max(conflito, key=lambda r: r.prioridade)
        elif self.estrategia == 'especificidade':
            return max(conflito, key=lambda r: len(r.condicoes))
        else:  # 'primeira'
            return conflito[0]

    def _execute(self, regra: Regra, instanciacao: tuple):
        """
        Fase EXECUTE: Aplica ações da regra.

        Parâmetros:
        -----------
        regra : Regra
            Regra a executar
        instanciacao : tuple
            Instanciação disparada (registrada para a refratariedade)

        Ações suportadas:
        - Adicionar fato: "ADD fato"
        - Remover fato: "REMOVE fato"
        - Conclusão: "CONCLUIR mensagem"
        """
        if self.debug:
            print(f"\n🔥 DISPARANDO: {regra.nome}")
            print(f"   Condições satisfeitas: {', '.join(regra.condicoes)}")

        for acao in regra.acoes:
            acao = acao.strip()

            if acao.startswith("ADD "):
                fato = acao[4:].strip()
                if self._inserir(fato):
                    self._novos.append(fato)
                if self.debug:
                    print(f"   → Adicionado: {fato}")

            elif acao.startswith("REMOVE "):
                fato = acao[7:].strip()
                if fato in self.memoria_trabalho:
                    self._removidos.append(fato)
                self.memoria_trabalho.discard(fato)
                self.timestamps.pop(fato, None)
                if self.debug:
                    print(f"   → Removido: {fato}")

            elif acao.startswith("CONCLUIR "):
                mensagem = acao[9:].strip()
                if self.debug:
                    print(f"   ✓ CONCLUSÃO: {mensagem}")

            else:
                # Ação genérica (adiciona como fato)
                if self._inserir(acao):
                    self._novos.append(acao)
                if self.debug:
                    print(f"   → {acao}")

        # Registra histórico e refratariedade (esta instanciação não poderá disparar de novo)
        self.historico_regras.append(regra.nome)
        self.regras_disparadas.add(instanciacao)

    def executar(self, max_iteracoes=100):
        """
        Executa o ciclo Reconhecer-Agir (Forward Chaining).

        Parâmetros:
        -----------
        max_iteracoes : int
            Limite de iterações para evitar loops infinitos
        """
        print("="*60)
        print("INICIANDO SISTEMA DE PRODUÇÃO")
        print("="*60)
        print(f"\nMemória de trabalho inicial: {self.memoria_trabalho}")
        print(f"Número de regras: {len(self.base_conhecimento)}")
        print(f"Estratégia de resolução: {self.estrategia}\n")

        iteracao = 0
        parou = False
        self.trace = []

        while iteracao < max_iteracoes:
            iteracao += 1

            if self.debug:
                print(f"\n{'─'*60}")
                print(f"CICLO {iteracao}")
                print(f"{'─'*60}")

            # MATCH: Encontra regras aplicáveis
            memoria_antes = sorted(self.memoria_trabalho)
            conflito = self._match()

            if not conflito:
                self.trace.append({'memoria': memoria_antes, 'conflito': [], 'escolhida': None,
                                   'novos': [], 'removidos': []})
                parou = True
                print(f"\n{'='*60}")
                print("SISTEMA PAROU: Nenhuma regra aplicável")
                print(f"{'='*60}")
                break

            if self.debug:
                print(f"\n📋 Regras aplicáveis: {[r.nome for r in conflito]}")

            # SELECT: Escolhe uma regra
            regra_escolhida = self._select(conflito)

            if self.debug:
                print(f"✓ Regra selecionada: {regra_escolhida.nome}")

            # EXECUTE: Aplica a regra (a instanciação é calculada ANTES de alterar a memória)
            self._novos, self._removidos = [], []
            self._execute(regra_escolhida, self._instanciacao(regra_escolhida))
            self.trace.append({'memoria': memoria_antes, 'conflito': [r.nome for r in conflito],
                               'escolhida': regra_escolhida.nome,
                               'novos': list(self._novos), 'removidos': list(self._removidos)})

        if not parou:
            print(f"\n⚠ AVISO: Limite de iterações ({max_iteracoes}) alcançado")

        self._exibir_resumo()

    def _exibir_resumo(self):
        """
        Exibe resumo da execução.
        """
        print(f"\n{'='*60}")
        print("RESUMO DA EXECUÇÃO")
        print(f"{'='*60}")
        print(f"\nMemória de trabalho final:")
        for fato in sorted(self.memoria_trabalho):
            print(f"  • {fato}")

        print(f"\nSequência de regras disparadas:")
        for i, nome_regra in enumerate(self.historico_regras, 1):
            print(f"  {i}. {nome_regra}")

        print(f"\nTotal de inferências: {len(self.historico_regras)}")
        print(f"{'='*60}\n")

# Exemplo: Sistema de classificação de animais
print("="*60)
print("EXEMPLO: CLASSIFICAÇÃO DE ANIMAIS")
print("="*60)

sistema = SistemaProducao(estrategia_resolucao='prioridade', debug=True)

# Base de Conhecimento
print("\n📚 Construindo Base de Conhecimento...\n")

sistema.adicionar_regra(
    nome="R1_Mamifero",
    condicoes=["tem_pelos", "amamenta"],
    acoes=["ADD eh_mamifero"],
    prioridade=5
)

sistema.adicionar_regra(
    nome="R2_Ave",
    condicoes=["tem_penas", "bota_ovos"],
    acoes=["ADD eh_ave"],
    prioridade=5
)

sistema.adicionar_regra(
    nome="R3_Carnivoro",
    condicoes=["eh_mamifero", "come_carne"],
    acoes=["ADD eh_carnivoro"],
    prioridade=3
)

sistema.adicionar_regra(
    nome="R4_Herbivoro",
    condicoes=["eh_mamifero", "come_plantas"],
    acoes=["ADD eh_herbivoro"],
    prioridade=3
)

sistema.adicionar_regra(
    nome="R5_Felino",
    condicoes=["eh_carnivoro", "tem_garras_retráteis"],
    acoes=["ADD eh_felino", "CONCLUIR Animal é um FELINO"],
    prioridade=1
)

sistema.adicionar_regra(
    nome="R6_Canino",
    condicoes=["eh_carnivoro", "late"],
    acoes=["ADD eh_canino", "CONCLUIR Animal é um CANINO"],
    prioridade=1
)

sistema.adicionar_regra(
    nome="R7_Equino",
    condicoes=["eh_herbivoro", "tem_cascos"],
    acoes=["ADD eh_equino", "CONCLUIR Animal é um EQUINO"],
    prioridade=1
)

# Fatos iniciais (exemplo: cachorro)
print("\n📝 Adicionando Fatos Iniciais...\n")
sistema.adicionar_fato("tem_pelos")
sistema.adicionar_fato("amamenta")
sistema.adicionar_fato("come_carne")
sistema.adicionar_fato("late")

# Executa sistema
sistema.executar()

#### 🔎 Como interpretar o resultado

A saída é o **traço de execução** do motor. Leia de cima para baixo:

1. **Construção da base:** as 7 regras são carregadas e os 4 fatos iniciais (`tem_pelos`, `amamenta`, `come_carne`, `late`) entram na memória de trabalho.
2. **Ciclo 1:** apenas **R1_Mamifero** tem todas as condições satisfeitas, então ela dispara e adiciona `eh_mamifero`.
3. **Ciclo 2:** o fato novo **habilita R3_Carnivoro**, que adiciona `eh_carnivoro`.
4. **Ciclo 3:** agora **R6_Canino** é aplicável e conclui **"Animal é um CANINO"**.
5. **Ciclo 4:** o conjunto de conflito está **vazio** e o sistema para.

- **Por que cada regra dispara só uma vez?** Por causa da **refratariedade**: depois que R1 dispara com os fatos `tem_pelos` e `amamenta`, essa mesma combinação não pode dispará-la de novo. Sem isso, R1 continuaria aplicável (seus fatos continuam na memória) e dispararia **para sempre**.
- **Por que R2, R4, R5 e R7 nunca disparam?** Suas condições nunca foram satisfeitas (não há `tem_penas`, `come_plantas` etc.). Uma base de conhecimento normalmente tem **muito mais regras do que as usadas em um caso**.
- **💡 Conceito-chave:** isto é **encadeamento progressivo**, guiado pelos dados: partimos dos fatos e derivamos tudo o que for possível, camada por camada (pelos → mamífero → carnívoro → canino).

> ⚠️ **Correção em relação à versão anterior deste notebook:** antes, a refratariedade usava o **estado inteiro da memória** como chave. Assim, sempre que um fato novo entrava, a mesma regra podia disparar de novo, e R1 disparava **4 vezes** em vez de uma. Agora a chave é a **instanciação** (regra + fatos que a satisfizeram), como no OPS5 e no CLIPS. Também foi corrigido o aviso de "limite de iterações" que aparecia mesmo quando o sistema parava normalmente.

### 3.1 Observações sobre a Implementação

**Pontos-chave da implementação acima:**

1. **Modularidade das Regras:** Cada regra é uma entidade independente. Adicionar nova regra não requer modificar código existente — apenas chamada de `adicionar_regra()`.

2. **Estratégias de Resolução de Conflitos:**
   - **Prioridade:** Permite controle explícito sobre ordem de disparo
   - **Especificidade:** Regras mais específicas (mais condições) disparam primeiro
   - **Primeira:** Ordem de definição (simples mas previsível)

3. **Refratariedade:** Sistema implementa *refraction* — uma mesma **instanciação** (a regra + os fatos exatos que a satisfizeram, identificados pelo momento em que entraram na memória) nunca dispara duas vezes. Previne *loops* infinitos e disparos redundantes.

4. **Transparência:** O traço de execução mostra claramente:
   - Quais regras foram disparadas
   - Por que foram escolhidas
   - Que mudanças causaram na memória

5. **Limitações desta Implementação:**
   - Não suporta variáveis (unificação)
   - Casamento de padrões é simples (igualdade de strings)
   - Sem suporte a quantificadores (∀, ∃)
   - Algoritmo de matching é ingênuo (O(n×m) onde n=regras, m=fatos)

**Aplicação Prática:**

No exemplo acima, o sistema classifica animais usando encadeamento para frente:
1. Fatos iniciais: tem_pelos, amamenta, come_carne, late
2. Sistema infere: eh_mamifero (R1)
3. Depois infere: eh_carnivoro (R3)
4. Finalmente conclui: eh_canino (R6)

Essa é a essência do **raciocínio data-driven** (dirigido por dados).

### 3.2 🎬 Animação: o ciclo reconhecer–agir passo a passo

A animação mostra, a cada ciclo, **o conjunto de conflito** (amarelo), **a regra escolhida** (verde) e **o fato novo** produzido (laranja). Regras com contorno preto já dispararam em ciclos anteriores.

In [ ]:
import contextlib, io

def criar_sistema_animais(fatos, estrategia='prioridade'):
    # Mesma base de conhecimento do exemplo anterior, sem mensagens de depuração.
    s = SistemaProducao(estrategia_resolucao=estrategia, debug=False)
    s.adicionar_regra("R1_Mamifero", ["tem_pelos", "amamenta"], ["ADD eh_mamifero"], 5)
    s.adicionar_regra("R2_Ave", ["tem_penas", "bota_ovos"], ["ADD eh_ave"], 5)
    s.adicionar_regra("R3_Carnivoro", ["eh_mamifero", "come_carne"], ["ADD eh_carnivoro"], 3)
    s.adicionar_regra("R4_Herbivoro", ["eh_mamifero", "come_plantas"], ["ADD eh_herbivoro"], 3)
    s.adicionar_regra("R5_Felino", ["eh_carnivoro", "tem_garras_retráteis"], ["ADD eh_felino"], 1)
    s.adicionar_regra("R6_Canino", ["eh_carnivoro", "late"], ["ADD eh_canino"], 1)
    s.adicionar_regra("R7_Equino", ["eh_herbivoro", "tem_cascos"], ["ADD eh_equino"], 1)
    for f in fatos:
        s.adicionar_fato(f)
    return s


def executar_silencioso(sistema, **kw):
    with contextlib.redirect_stdout(io.StringIO()):
        sistema.executar(**kw)
    return sistema


def animar_ciclos(sistema, titulo=''):
    # Esquerda: regras (cinza = não aplicável, amarelo = conjunto de conflito, verde = disparada).
    # Direita: memória de trabalho (azul = fato presente, laranja = fato novo neste ciclo).
    regras, trace = sistema.base_conhecimento, sistema.trace
    fatos = sorted(set().union(*[set(t['memoria']) | set(t['novos']) for t in trace]))
    fig, (axr, axm) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={'width_ratios': [1.6, 1]})
    fig.subplots_adjust(top=0.78)

    def quadro(k):
        t = trace[k]
        disparadas_antes = {trace[j]['escolhida'] for j in range(k)}
        axr.clear(); axm.clear(); axr.axis('off'); axm.axis('off')
        for i, r in enumerate(regras):
            if r.nome == t['escolhida']:
                cor = '#81c784'
            elif r.nome in t['conflito']:
                cor = CORES['amarelo']
            else:
                cor = '#eeeeee'
            axr.add_patch(plt.Rectangle((0, -i), 1, 0.85, color=cor,
                                        ec='k' if r.nome in disparadas_antes else 'none'))
            texto = f"{r.nome}: {' ∧ '.join(r.condicoes)} → {', '.join(a.replace('ADD ', '') for a in r.acoes)}"
            axr.text(0.01, -i + 0.42, texto, va='center', fontsize=9)
        axr.set_xlim(0, 1); axr.set_ylim(-len(regras) + 0.5, 1.2)
        acao = (f"dispara {t['escolhida']} → adiciona {t['novos'] or 'nada (já existia)'}"
                if t['escolhida'] else 'conjunto de conflito VAZIO → o sistema para')
        axr.set_title(f"Ciclo {k + 1} | conflito = {t['conflito']}\n{acao}", fontsize=10)
        presentes = set(t['memoria']) | set(t['novos'])
        for i, f in enumerate(fatos):
            x, y = i % 2, -(i // 2)
            cor = CORES['laranja'] if f in t['novos'] else ('#90caf9' if f in presentes else 'white')
            axm.add_patch(plt.Rectangle((x, y), 0.95, 0.8, color=cor, ec='gray', ls='-' if f in presentes else ':'))
            axm.text(x + 0.47, y + 0.4, f, ha='center', va='center', fontsize=9,
                     color='k' if f in presentes else '#aaaaaa')
        axm.set_xlim(0, 2); axm.set_ylim(-(len(fatos) // 2) - 0.5, 1.2)
        axm.set_title('Memória de trabalho', fontsize=10)
        fig.suptitle(titulo, fontsize=12, y=0.99)

    return animation.FuncAnimation(fig, quadro, frames=len(trace), interval=1300)


cachorro = executar_silencioso(criar_sistema_animais(['tem_pelos', 'amamenta', 'come_carne', 'late']))
print('Sequência de regras disparadas:', cachorro.historico_regras)
mostrar_animacao(animar_ciclos(cachorro, 'Forward chaining: classificando um cachorro'))

#### 🔎 Como interpretar a animação

| Cor | Significado |
|---|---|
| ⬜ Cinza claro | Regra **não aplicável** (alguma condição é falsa) |
| 🟨 Amarelo | Regra no **conjunto de conflito**: aplicável, mas não escolhida |
| 🟩 Verde | Regra **escolhida** e disparada neste ciclo |
| ▭ Contorno preto | Regra que **já disparou** antes (bloqueada pela refratariedade) |
| 🟧 Laranja (memória) | Fato **novo**, criado neste ciclo |

- **O que observar:** a "onda" de inferência descendo pela base. Cada fato laranja aparece **um ciclo antes** de a regra que depende dele ficar amarela/verde. É o encadeamento acontecendo.
- Repare que, neste exemplo, o conjunto de conflito tem **uma só regra por ciclo**: não há disputa, e a estratégia de resolução não faz diferença. Na Seção 6.2.5 veremos um caso em que várias regras competem e a escolha muda o resultado.
- **Último quadro:** o conjunto de conflito está **vazio** e o sistema para. A memória final contém os 4 fatos de entrada e os 3 derivados.

### 3.3 Mesmas regras, fatos diferentes

A grande vantagem da **separação entre conhecimento e controle**: a base de regras não muda — apenas os **fatos iniciais**.

In [ ]:
# Os MESMOS dados de regras, outro animal: só mudam os FATOS iniciais
cavalo = executar_silencioso(criar_sistema_animais(['tem_pelos', 'amamenta', 'come_plantas', 'tem_cascos']))
print('Sequência de regras disparadas:', cavalo.historico_regras)
print('Memória final:', sorted(cavalo.memoria_trabalho))

pinguim = executar_silencioso(criar_sistema_animais(['tem_penas', 'bota_ovos', 'nada']))
print('\nPinguim → regras disparadas:', pinguim.historico_regras)
print('Memória final:', sorted(pinguim.memoria_trabalho))

#### 🔎 Como interpretar o resultado

- **Cavalo:** os fatos `come_plantas` e `tem_cascos` levaram ao caminho **R1 → R4 → R7** (mamífero → herbívoro → **equino**). As regras são **as mesmas** do cachorro; só os dados mudaram.
- **Pinguim:** só **R2_Ave** disparou. O sistema conclui `eh_ave` e **para**, porque não existe nenhuma regra que use `eh_ave` para chegar a uma espécie.
- **O que isso ensina?**
  1. **Separação conhecimento × controle:** o mesmo motor e a mesma base resolvem casos diferentes, sem mudar uma linha de código.
  2. **Fragilidade (*brittleness*):** o sistema só sabe o que está nas regras. Diante de um pinguim, ele não diz "não sei"; apenas **para** com uma conclusão parcial. Para resolver, um engenheiro de conhecimento precisaria **adicionar regras** (por exemplo, `SE eh_ave E nada ENTÃO eh_pinguim`).
- **💡 Experimente:** adicione essa regra em `criar_sistema_animais` e rode de novo o caso do pinguim.

---

## 4. Encadeamento para Trás (Backward Chaining)

Enquanto **Forward Chaining** parte dos fatos e deriva conclusões, **Backward Chaining** parte de um **objetivo** (goal) e tenta encontrar fatos que o suportem.

### 4.1 Conceito

**Backward Chaining** é uma estratégia de inferência **goal-driven** (dirigida por objetivos). O sistema:

1. Começa com uma **meta** a provar
2. Procura regras que **concluem** essa meta
3. Para cada regra encontrada, tenta provar suas **condições** (que se tornam submetas)
4. Recursivamente, repete o processo para cada submeta
5. Sucesso quando todas as submetas são provadas ou encontradas na base de fatos

### 4.2 Algoritmo

```
função BACKWARD_CHAINING(base_conhecimento, objetivo, fatos):
    SE objetivo ∈ fatos ENTÃO
        RETORNA verdadeiro  // Objetivo já é fato conhecido
    
    PARA CADA regra EM base_conhecimento:
        SE conclusão(regra) = objetivo ENTÃO
            todas_condicoes_satisfeitas ← verdadeiro
            
            PARA CADA condição EM condicoes(regra):
                SE NOT BACKWARD_CHAINING(base_conhecimento, condição, fatos) ENTÃO
                    todas_condicoes_satisfeitas ← falso
                    SAIR do loop
            
            SE todas_condicoes_satisfeitas ENTÃO
                RETORNA verdadeiro
    
    RETORNA falso  // Objetivo não pode ser provado
```

### 4.3 Características

**Vantagens:**
- **Eficiente para problemas diagnósticos:** Não explora todo espaço de fatos
- **Focado:** Apenas investiga caminhos relevantes ao objetivo
- **Natural para consultas:** "Este animal é um felino?"

**Desvantagens:**
- Pode ser ineficiente se múltiplos objetivos precisam ser verificados
- Pode repetir cálculos (requer **memoização** para eficiência)
- Menos intuitivo para sistemas reativos

### 4.4 Exemplo Conceitual

**Base de Conhecimento:**
```
R1: SE tem_penas E bota_ovos ENTÃO eh_ave
R2: SE eh_ave E nao_voa ENTÃO eh_pinguim
R3: SE eh_ave E voa_alto ENTÃO eh_aguia
```

**Fatos:** `{tem_penas, bota_ovos, nao_voa}`

**Objetivo:** Provar `eh_pinguim`

**Raciocínio Backward:**
1. Objetivo: `eh_pinguim` → Encontra R2 que conclui isso
2. Submetas de R2: `eh_ave`, `nao_voa`
   - `nao_voa` está nos fatos ✓
   - `eh_ave` não está → Procura regra que conclui isso
3. Encontra R1 que conclui `eh_ave`
4. Submetas de R1: `tem_penas`, `bota_ovos`
   - Ambos estão nos fatos ✓
5. Logo, `eh_ave` é provado ✓
6. Logo, `eh_pinguim` é provado ✓

### 4.5 Implementação de Backward Chaining

In [ ]:
class SistemaBackwardChaining:
    """
    Sistema de Produção com Encadeamento para Trás (Backward Chaining).
    
    Estratégia goal-driven: parte do objetivo e busca fatos que o suportem.
    """
    
    def __init__(self, debug=True):
        self.base_conhecimento: List[Regra] = []
        self.base_fatos: Set[str] = set()
        self.debug = debug
        self.tentativas = []  # Rastreamento de tentativas
        self.profundidade = 0  # Para indentação do traço
        self.memoizacao = {}  # Cache de objetivos já provados
    
    def adicionar_regra(self, nome: str, condicoes: List[str], 
                       conclusao: str, prioridade: int = 0):
        """
        Adiciona regra à base de conhecimento.
        
        Nota: No backward chaining, regras têm UMA conclusão (não múltiplas ações).
        """
        # Adaptamos Regra: conclusão vai em 'acoes' como único elemento
        regra = Regra(nome, condicoes, [conclusao], prioridade)
        self.base_conhecimento.append(regra)
        if self.debug:
            print(f"✓ Regra adicionada: {nome}")
    
    def adicionar_fato(self, fato: str):
        """
        Adiciona fato à base de fatos.
        """
        self.base_fatos.add(fato)
        if self.debug:
            print(f"✓ Fato adicionado: {fato}")
    
    def _indent(self) -> str:
        """Retorna string de indentação baseada na profundidade."""
        return "  " * self.profundidade
    
    def provar(self, objetivo: str) -> bool:
        """
        Tenta provar um objetivo usando backward chaining.
        
        Parâmetros:
        -----------
        objetivo : str
            Fato a ser provado
        
        Retorna:
        --------
        True se objetivo pode ser provado, False caso contrário
        """
        if self.debug:
            print(f"\n{self._indent()}🎯 Tentando provar: {objetivo}")
        
        self.tentativas.append(objetivo)
        
        # Verifica memoização
        if objetivo in self.memoizacao:
            resultado = self.memoizacao[objetivo]
            if self.debug:
                print(f"{self._indent()}   [Cache] Já provado: {resultado}")
            return resultado
        
        # Caso base: objetivo já é fato conhecido
        if objetivo in self.base_fatos:
            if self.debug:
                print(f"{self._indent()}   ✓ Fato encontrado na base!")
            self.memoizacao[objetivo] = True
            return True
        
        # Busca recursiva: procura regras que concluem o objetivo
        self.profundidade += 1
        
        for regra in self.base_conhecimento:
            # Verifica se regra conclui o objetivo
            conclusao_regra = regra.acoes[0]  # No BC, só uma conclusão
            
            if conclusao_regra == objetivo:
                if self.debug:
                    print(f"{self._indent()}📋 Encontrada regra: {regra.nome}")
                    print(f"{self._indent()}   Precisa provar: {', '.join(regra.condicoes)}")
                
                # Tenta provar todas as condições da regra
                todas_provadas = True
                
                for condicao in regra.condicoes:
                    if not self.provar(condicao):
                        todas_provadas = False
                        if self.debug:
                            print(f"{self._indent()}   ✗ Não conseguiu provar: {condicao}")
                        break
                
                if todas_provadas:
                    if self.debug:
                        print(f"{self._indent()}   ✓ Todas condições provadas! {regra.nome} aplica!")
                    self.profundidade -= 1
                    self.memoizacao[objetivo] = True
                    return True
        
        # Nenhuma regra conseguiu provar o objetivo
        if self.debug:
            print(f"{self._indent()}   ✗ Não foi possível provar {objetivo}")
        
        self.profundidade -= 1
        self.memoizacao[objetivo] = False
        return False
    
    def consultar(self, objetivo: str):
        """
        Interface amigável para consultas.
        """
        print("="*60)
        print("BACKWARD CHAINING — Provador de Objetivos")
        print("="*60)
        print(f"\nBase de fatos: {sorted(self.base_fatos)}")
        print(f"Número de regras: {len(self.base_conhecimento)}")
        print(f"\nObjetivo a provar: {objetivo}")
        print("="*60)
        
        self.tentativas = []
        self.profundidade = 0
        self.memoizacao = {}
        
        resultado = self.provar(objetivo)
        
        print("\n" + "="*60)
        if resultado:
            print(f"✓ SUCESSO: '{objetivo}' foi PROVADO")
        else:
            print(f"✗ FALHA: '{objetivo}' NÃO pode ser provado")
        print("="*60)
        print(f"\nTotal de tentativas: {len(self.tentativas)}")
        print(f"Submetas exploradas: {self.tentativas}")
        print("="*60 + "\n")
        
        return resultado


# Exemplo: Diagnóstico médico simplificado
print("="*60)
print("EXEMPLO: DIAGNÓSTICO MÉDICO COM BACKWARD CHAINING")
print("="*60)

sistema_bc = SistemaBackwardChaining(debug=True)

# Base de Conhecimento (regras diagnósticas)
print("\n📚 Construindo Base de Conhecimento...\n")

sistema_bc.adicionar_regra(
    nome="R1_Infeccao",
    condicoes=["febre", "dor_corpo"],
    conclusao="tem_infeccao"
)

sistema_bc.adicionar_regra(
    nome="R2_Gripe",
    condicoes=["tem_infeccao", "tosse", "coriza"],
    conclusao="tem_gripe"
)

sistema_bc.adicionar_regra(
    nome="R3_Pneumonia",
    condicoes=["tem_infeccao", "dificuldade_respirar", "tosse_persistente"],
    conclusao="tem_pneumonia"
)

sistema_bc.adicionar_regra(
    nome="R4_Alergia",
    condicoes=["espirros", "coceira"],
    conclusao="tem_alergia"
)

sistema_bc.adicionar_regra(
    nome="R5_Resfriado",
    condicoes=["coriza", "espirros"],
    conclusao="tem_resfriado"
)

# Fatos observados (sintomas do paciente)
print("\n📝 Sintomas Observados no Paciente...\n")
sistema_bc.adicionar_fato("febre")
sistema_bc.adicionar_fato("dor_corpo")
sistema_bc.adicionar_fato("tosse")
sistema_bc.adicionar_fato("coriza")

# Consultas diagnósticas
print("\n" + "="*60)
print("REALIZANDO DIAGNÓSTICOS")
print("="*60)

# Teste 1: Verificar se paciente tem gripe
sistema_bc.consultar("tem_gripe")

# Teste 2: Verificar se paciente tem pneumonia
sistema_bc.consultar("tem_pneumonia")

# Teste 3: Verificar se paciente tem resfriado
sistema_bc.consultar("tem_resfriado")

#### 🔎 Como interpretar o resultado

O traço tem **indentação**: cada nível mais à direita é uma **submeta** criada para provar o nível acima.

| Consulta | Resultado | Caminho de raciocínio |
|---|---|---|
| `tem_gripe` | ✅ provado | `tem_gripe` ← R2 precisa de `tem_infeccao`, `tosse`, `coriza` → `tem_infeccao` ← R1 precisa de `febre` e `dor_corpo` (fatos) |
| `tem_pneumonia` | ❌ falhou | `tem_infeccao` foi provada, mas `dificuldade_respirar` **não é fato e nenhuma regra a conclui** |
| `tem_resfriado` | ❌ falhou | `coriza` é fato, mas `espirros` não |

- **Repare no número de tentativas:** na pneumonia foram só **5** submetas. Assim que `dificuldade_respirar` falhou, o motor **nem tentou** `tosse_persistente`. Uma conjunção (E) já é falsa quando uma parte falha. É o **curto-circuito**, que economiza trabalho.
- **Por que nenhuma regra de alergia foi examinada?** O *backward chaining* só olha regras cuja **conclusão** é a meta atual. R4_Alergia nunca é relevante para provar gripe, pneumonia ou resfriado. É por isso que ele é **focado**.
- **⚠️ Detalhe importante:** "não provado" **não significa falso**. Significa "não consegui provar com os fatos que tenho". O paciente pode ter pneumonia, mas ninguém informou a dificuldade respiratória. É a **hipótese do mundo fechado**: o que não está na base é tratado como falso. Sistemas reais como o MYCIN, em vez de falhar, **perguntavam** ao usuário.

### 4.6 Observações sobre Backward Chaining

**Diferenças Fundamentais do Forward Chaining:**

1. **Direção do Raciocínio:**
   - **Forward:** Dados → Conclusões (bottom-up)
   - **Backward:** Objetivo → Fatos necessários (top-down)

2. **Estrutura das Regras:**
   - **Forward:** Regras podem ter múltiplas ações
   - **Backward:** Regras tipicamente têm uma conclusão

3. **Uso de Recursão:**
   - Backward chaining é naturalmente recursivo
   - Cada condição vira submeta a provar

4. **Quando Usar:**
   - **Forward:** Sistemas reativos, monitoramento, alertas, derivação de todos os fatos possíveis
   - **Backward:** Diagnóstico, consultas específicas, verificação de hipóteses

**Exemplo no Diagnóstico Médico:**

O sistema tenta provar "tem_gripe":
1. Encontra R2 que conclui "tem_gripe"
2. Precisa provar: tem_infeccao, tosse, coriza
3. "tosse" e "coriza" são fatos conhecidos ✓
4. "tem_infeccao" não é fato → Busca regra
5. Encontra R1 que conclui "tem_infeccao"
6. Precisa provar: febre, dor_corpo
7. Ambos são fatos conhecidos ✓
8. Logo, "tem_infeccao" é provado ✓
9. Logo, "tem_gripe" é provado ✓

**Otimização — Memoização:**

A implementação usa cache (`memoizacao`) para evitar reprovar objetivos:
- Sem cache: complexidade pode ser exponencial
- Com cache: cada objetivo é provado no máximo uma vez
- Essencial para eficiência em bases grandes

### 4.7 🎬 Animação: a árvore de prova (com *backtracking*)

O *backward chaining* constrói uma **árvore E** (*AND-tree*): para provar um nó, é preciso provar **todos** os filhos. Cores:
🟨 tentando provar · 🟩 provado · 🟥 falhou.

A base abaixo tem **duas regras** para `eh_ave`. A primeira (`R1`) exige `voa` e vai **falhar** — o motor então **volta atrás** e tenta `R2`.

```
R1: SE voa E bota_ovos              ENTÃO eh_ave
R2: SE tem_penas E bota_ovos        ENTÃO eh_ave
R3: SE eh_ave E nada_bem E nao_voa  ENTÃO eh_pinguim
R4: SE eh_ave E voa_alto            ENTÃO eh_aguia
Fatos: tem_penas, bota_ovos, nao_voa, nada_bem
```

In [ ]:
def arvore_de_prova(regras, fatos, objetivo):
    # Backward chaining que REGISTRA cada passo para podermos desenhar a árvore de prova.
    # regras: lista de (nome, [condições], conclusão)
    nos, eventos = [], []

    def provar(meta, pai):
        id_ = len(nos)
        nos.append({'meta': meta, 'pai': pai})
        eventos.append(('abre', id_, None))
        if meta in fatos:
            eventos.append(('ok', id_, 'fato')); return True
        for nome, conds, concl in regras:
            if concl == meta:
                eventos.append(('regra', id_, nome))
                if all(provar(c, id_) for c in conds):     # 'all' para no primeiro fracasso
                    eventos.append(('ok', id_, nome)); return True
        eventos.append(('falha', id_, None)); return False

    resultado = provar(objetivo, None)
    return nos, eventos, resultado


def posicoes_arvore(nos):
    filhos = {i: [] for i in range(len(nos))}
    for i, n in enumerate(nos):
        if n['pai'] is not None:
            filhos[n['pai']].append(i)
    pos, prox_x = {}, [0]

    def dfs(i, prof):
        if not filhos[i]:
            pos[i] = (prox_x[0], -prof); prox_x[0] += 1
        else:
            for f in filhos[i]:
                dfs(f, prof + 1)
            pos[i] = (np.mean([pos[f][0] for f in filhos[i]]), -prof)
    dfs(0, 0)
    return pos


def animar_prova(regras, fatos, objetivo):
    nos, eventos, resultado = arvore_de_prova(regras, fatos, objetivo)
    pos = posicoes_arvore(nos)
    fig, ax = plt.subplots(figsize=(11, 5))
    cores_status = {'abre': CORES['amarelo'], 'ok': '#81c784', 'falha': '#ef9a9a'}

    def quadro(k):
        ax.clear(); ax.axis('off')
        status, regra_usada, visiveis = {}, {}, set()
        for tipo, i, info in eventos[:k + 1]:
            visiveis.add(i)
            if tipo == 'regra':
                regra_usada[i] = info
            else:
                status[i] = tipo
        for i in visiveis:
            p = nos[i]['pai']
            if p is not None:
                ax.plot([pos[p][0], pos[i][0]], [pos[p][1], pos[i][1]], 'k-', lw=1, zorder=1)
        for i in visiveis:
            x, y = pos[i]
            ax.text(x, y, nos[i]['meta'], ha='center', va='center', fontsize=10, zorder=2,
                    bbox=dict(boxstyle='round', fc=cores_status[status[i]], ec='k'))
            if i in regra_usada:
                ax.text(x, y + 0.28, f'via {regra_usada[i]}', ha='center', fontsize=8, color='#555')
        tipo, i, info = eventos[k]
        msg = {'abre': f'tentando provar "{nos[i]["meta"]}"',
               'regra': f'"{nos[i]["meta"]}" não é fato: tentando a regra {info}',
               'ok': f'"{nos[i]["meta"]}" PROVADO' + (' (é um fato)' if info == 'fato' else f' (pela regra {info})'),
               'falha': f'"{nos[i]["meta"]}" FALHOU: não é fato e nenhuma regra funciona'}[tipo]
        ax.set_title(f'Passo {k + 1}/{len(eventos)}: {msg}', fontsize=11)
        xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
        ax.set_xlim(min(xs) - 0.8, max(xs) + 0.8); ax.set_ylim(min(ys) - 0.6, 0.7)

    print(f'Objetivo "{objetivo}":', 'PROVADO ✅' if resultado else 'NÃO PROVADO ❌')
    return animation.FuncAnimation(fig, quadro, frames=len(eventos), interval=900)


# Base com DUAS regras para "eh_ave": a primeira vai falhar e o motor terá de voltar atrás (backtracking)
regras_aves = [
    ('R1', ['voa', 'bota_ovos'], 'eh_ave'),
    ('R2', ['tem_penas', 'bota_ovos'], 'eh_ave'),
    ('R3', ['eh_ave', 'nada_bem', 'nao_voa'], 'eh_pinguim'),
    ('R4', ['eh_ave', 'voa_alto'], 'eh_aguia'),
]
fatos_aves = {'tem_penas', 'bota_ovos', 'nao_voa', 'nada_bem'}
mostrar_animacao(animar_prova(regras_aves, fatos_aves, 'eh_pinguim'))

#### 🔎 Como interpretar a animação

Acompanhe a construção da árvore **passo a passo** (use ⏭):

1. A raiz `eh_pinguim` (🟨) não é fato, então o motor usa **R3**, que precisa de `eh_ave`, `nada_bem` e `nao_voa`.
2. Para `eh_ave` existem **duas** regras. O motor tenta **R1** primeiro: precisa de `voa`, que **falha** (🟥). Pelo curto-circuito, `bota_ovos` nem é testado nesse ramo.
3. **Backtracking:** o motor **volta** e tenta **R2** para o mesmo nó: `tem_penas` ✅ e `bota_ovos` ✅, então `eh_ave` fica 🟩.
4. Em seguida `nada_bem` e `nao_voa` são fatos 🟩, e `eh_pinguim` é **PROVADO**.

- **Por que o nó `voa` continua vermelho na árvore final?** Ele registra uma **tentativa que não deu certo**. A árvore mostra todo o **esforço de busca**, não só a prova final. A prova "limpa" é o caminho verde.
- **💡 Conceito-chave:** *backward chaining* é uma **busca em profundidade** em uma árvore E/OU: os filhos de um nó-meta via **uma** regra estão ligados por **E** (todos precisam ser provados), e as **regras alternativas** para a mesma meta formam um **OU** (basta uma funcionar). É exatamente o mecanismo de execução da linguagem **Prolog**.

In [ ]:
# Um objetivo que NÃO pode ser provado: veja a falha "subir" pela árvore
mostrar_animacao(animar_prova(regras_aves, fatos_aves, 'eh_aguia'))

#### 🔎 Como interpretar a animação

- O motor consegue provar `eh_ave` (mesmo caminho R1 ✗ → R2 ✓ de antes), mas **R4** também precisa de `voa_alto`, que não é fato e não é conclusão de regra nenhuma: 🟥.
- A falha **sobe** pela árvore: como `eh_aguia` só tem uma regra e ela falhou, a raiz fica vermelha.
- **Compare com o forward chaining:** para responder "é uma águia?", o forward teria de derivar **tudo** primeiro e depois verificar se `eh_aguia` apareceu. O backward investigou **apenas** o que era relevante para essa pergunta.

---

## 5. Comparação: Forward vs. Backward Chaining

### 5.1 Tabela Comparativa

| **Aspecto** | **Forward Chaining** | **Backward Chaining** |
|---|---|---|
| **Direção** | Dados → Conclusões (bottom-up) | Objetivo → Premissas (top-down) |
| **Estratégia** | Data-driven (dirigido por dados) | Goal-driven (dirigido por objetivos) |
| **Ponto de Partida** | Fatos conhecidos | Objetivo a provar |
| **Processo** | Deriva todas conclusões possíveis | Busca fatos que suportam objetivo |
| **Algoritmo** | Iterativo (ciclo reconhecer-agir) | Recursivo (prova por decomposição) |
| **Espaço Explorado** | Pode explorar fatos irrelevantes | Focado apenas em caminhos relevantes |
| **Eficiência** | Menos eficiente para consultas pontuais | Mais eficiente para objetivos específicos |
| **Uso Típico** | Sistemas reativos, monitoramento, alertas | Diagnóstico, consultas, verificação |
| **Transparência** | Fácil seguir traço de execução | Pode ser complexo com recursão profunda |
| **Completude** | Deriva todos os fatos deriváveis | Só prova o que é perguntado |
| **Exemplo Clássico** | CLIPS, OPS5, sistemas de alarme | MYCIN, sistemas especialistas médicos |
| **Vantagens** | • Descobre todos os fatos<br>• Bom para múltiplos objetivos<br>• Simples de implementar | • Eficiente para metas específicas<br>• Não desperdiça tempo<br>• Natural para consultas |
| **Desvantagens** | • Pode ser ineficiente<br>• Deriva fatos desnecessários<br>• Pode ter muitos conflitos | • Pode reprovar submetas (sem cache)<br>• Menos intuitivo<br>• Complexo debugar |

### 5.2 Exemplo Ilustrativo

*(Números hipotéticos, apenas para ilustrar a ordem de grandeza — na Seção 5.4 medimos isso de verdade.)*

**Cenário:** Sistema com 100 regras e 50 fatos iniciais.

**Tarefa:** Verificar se "paciente_tem_doença_X" é verdadeiro.

**Forward Chaining:**
- Aplica todas as regras possíveis
- Deriva centenas de conclusões
- Eventualmente chega em "paciente_tem_doença_X"
- Tempo: 100 ms
- Regras disparadas: 45
- Conclusões derivadas: 200+

**Backward Chaining:**
- Começa com objetivo "paciente_tem_doença_X"
- Busca regras que concluem isso (encontra 2)
- Tenta provar condições dessas regras
- Tempo: 15 ms
- Regras exploradas: 8
- Submetas exploradas: 12

**Conclusão:** Para consultas pontuais, backward chaining é significativamente mais eficiente.

### 5.3 Sistemas Híbridos

Sistemas modernos frequentemente combinam ambas estratégias:

1. **Forward para fatos estáveis:** Deriva conhecimento geral
2. **Backward para consultas:** Responde perguntas específicas

**Exemplo:** Sistema médico pode:
- Usar forward chaining para categorizar sintomas gerais
- Usar backward chaining para confirmar diagnósticos específicos

Essa abordagem híbrida combina o melhor dos dois mundos.

### 5.4 Experimento: *forward* × *backward* em uma base grande

Vamos **medir** em vez de supor. Geramos automaticamente uma base com **240 regras em 5 camadas** (cada regra conclui um fato de uma camada a partir de 2 fatos da camada anterior) e comparamos:

- **Forward:** deriva **todos** os fatos possíveis, uma única vez;
- **Backward:** responde **uma pergunta** específica, visitando só o necessário.

In [ ]:
import random

def gerar_base(n_niveis=5, fatos_por_nivel=20, regras_por_nivel=60, semente=0):
    # Base sintética em camadas: cada regra do nível L conclui um fato do nível L a partir de 2 fatos do nível L-1.
    rng = random.Random(semente)
    nome = lambda nivel, i: f'f{nivel}_{i}'
    regras = []
    for nivel in range(1, n_niveis):
        for k in range(regras_por_nivel):
            conds = [nome(nivel - 1, i) for i in rng.sample(range(fatos_por_nivel), 2)]
            regras.append((f'R{nivel}_{k}', conds, nome(nivel, rng.randrange(fatos_por_nivel))))
    iniciais = {nome(0, i) for i in range(fatos_por_nivel) if rng.random() < 0.5}
    return regras, iniciais


def forward_contando(regras, fatos):
    fatos, testes, mudou = set(fatos), 0, True
    while mudou:
        mudou = False
        for nome, conds, concl in regras:
            testes += 1                                  # cada regra testada conta 1
            if concl not in fatos and all(c in fatos for c in conds):
                fatos.add(concl); mudou = True
    return fatos, testes


def backward_contando(regras, fatos, objetivo):
    por_conclusao = {}
    for r in regras:
        por_conclusao.setdefault(r[2], []).append(r)
    memo, contagem = {}, {'submetas': 0, 'testes': 0}

    def provar(meta):
        if meta in memo:
            return memo[meta]
        contagem['submetas'] += 1
        memo[meta] = False                               # evita ciclos
        if meta in fatos:
            memo[meta] = True; return True
        for nome, conds, concl in por_conclusao.get(meta, []):
            contagem['testes'] += 1
            if all(provar(c) for c in conds):
                memo[meta] = True; return True
        return False

    return provar(objetivo), contagem


regras_g, iniciais_g = gerar_base()
todos_fatos, testes_fw = forward_contando(regras_g, iniciais_g)
derivados = todos_fatos - iniciais_g
print(f'Base sintética: {len(regras_g)} regras, {len(iniciais_g)} fatos iniciais')
print(f'FORWARD : {testes_fw} testes de regra, {len(derivados)} fatos novos derivados (TODOS os possíveis)\n')

consultas = sorted(f for f in derivados if f.startswith('f4_'))[:5] + ['f4_inexistente']
linhas = []
for q in consultas:
    ok, c = backward_contando(regras_g, iniciais_g, q)
    linhas.append((q, ok, c['submetas'], c['testes']))
    print(f'BACKWARD: consulta {q:<15} → {"provado " if ok else "falhou  "} | '
          f'{c["submetas"]:3d} submetas visitadas | {c["testes"]:3d} regras testadas')

fig, ax = plt.subplots(figsize=(10, 3.8))
nomes = [l[0] for l in linhas]
ax.bar(nomes, [l[3] for l in linhas], color=CORES['verde'], label='backward: regras testadas por consulta')
ax.axhline(testes_fw, color=CORES['vermelho'], ls='--', label=f'forward: {testes_fw} testes (uma vez, deriva tudo)')
ax.set_yscale('log'); ax.set_ylabel('nº de testes (escala log)'); ax.legend(fontsize=9)
ax.set_title('Custo de responder UMA pergunta: forward × backward')
plt.xticks(rotation=15); plt.show()

#### 🔎 Como interpretar o resultado

- **Forward:** testou **480** regras (2 passagens pelas 240) e derivou **62 fatos novos**, ou seja, **tudo** o que é possível concluir.
- **Backward:** para responder a **uma** pergunta, visitou de **28 a 52 submetas** e testou de **21 a 51 regras**, uma ordem de grandeza a menos. E a consulta `f4_inexistente` foi descartada com **0 testes**, porque nenhuma regra conclui esse fato.
- **Como ler o gráfico** (escala logarítmica): as barras verdes (custo de cada consulta *backward*) ficam bem abaixo da linha vermelha (custo do *forward*).
- **Então o backward sempre ganha? Não.** O *forward* paga uma vez e responde **todas** as perguntas. Se precisássemos consultar os 62 fatos derivados, o *backward* faria 62 buscas (com repetição de trabalho entre elas, a menos que guarde o cache entre consultas). A regra prática:

| Situação | Melhor estratégia |
|---|---|
| **Poucas perguntas** específicas ("o paciente tem pneumonia?") | *Backward* |
| **Monitorar** tudo o que acontece / reagir a eventos | *Forward* |
| Dados chegam aos poucos e decisões devem ser imediatas | *Forward* |
| Fatos são caros de obter (exames, perguntas ao usuário) | *Backward*: só pergunta o necessário |

- **⚠️ Observação:** neste experimento as regras estão listadas **na ordem das camadas**, o que favorece o *forward* (bastam 2 passagens). Em um ciclo reconhecer–agir real, que dispara uma regra por vez e refaz o *match*, o custo do *forward* seria bem maior, como mostra o experimento da Seção 6.1.7.

---

## 6. Tópicos Avançados

### 6.1 O Algoritmo RETE

O **algoritmo RETE** (do latim *"rete"* = rede) foi desenvolvido por **Charles Forgy** em 1979 e revolucionou a eficiência dos sistemas de produção.

#### 6.1.1 Problema de Desempenho

Em sistemas ingênuos (como nossa implementação básica), a fase **MATCH** tem complexidade:

$$O(R \times F \times C)$$

Onde:
- $R$ = número de regras
- $F$ = número de fatos na memória de trabalho
- $C$ = número médio de condições por regra

**Problema:** A cada ciclo, sistema recompara **todas** as regras com **todos** os fatos — mesmo que apenas 1 fato tenha mudado!

**Exemplo:** Sistema com 1000 regras e 5000 fatos:
- Comparações por ciclo: 1000 × 5000 × 3 = **15 milhões de comparações**
- Se adicionar 1 fato: ainda faz 15 milhões de comparações

#### 6.1.2 Princípio do RETE

RETE explora **temporalidade** (temporal redundancy):

> "Entre dois ciclos sucessivos, a memória de trabalho muda muito pouco."

**Insight:** Não é necessário recomputar tudo — apenas atualizar o que mudou!

#### 6.1.3 Estrutura do RETE

RETE compila regras em uma **rede de nós** (rede RETE):

```
                    REGRAS
                      |
              ┌───────┴───────┐
              ↓               ↓
         [Nó Raiz]       [Nó Raiz]
              |               |
        ┌─────┴─────┐   ┌─────┴─────┐
        ↓           ↓   ↓           ↓
    [Nó Tipo]   [Nó Tipo] [Nó Tipo] ...
        |           |       |
    [Nó Filtro] [Nó Filtro] ...
        |           |
    [Nó Join]   [Nó Join]
        |           |
    [Nó Terminal] [Nó Terminal]
```

**Tipos de Nós:**

1. **Nós de Tipo (Alpha Nodes):** Testam condições sobre fatos individuais
   - Ex: "tipo = mamífero", "temperatura > 25"

2. **Nós de Junção (Beta Nodes):** Testam condições entre múltiplos fatos
   - Ex: "fato1.temperatura = fato2.setpoint"

3. **Nós Terminais (Production Nodes):** Representam regras completamente casadas

#### 6.1.4 Funcionamento

**Fase de Compilação (uma vez):**
1. Cada regra é compilada em caminhos na rede
2. Condições comuns são compartilhadas entre regras

**Fase de Execução (a cada ciclo):**
1. Quando fato é adicionado/removido, propaga pela rede
2. Apenas nós afetados são atualizados (não toda rede)
3. Nós terminais ativados indicam regras aplicáveis

**Exemplo:**

```
Regra A: SE animal(X) E mamifero(X) ENTÃO carnivoro(X)
Regra B: SE animal(X) E mamifero(X) E come_carne(X) ENTÃO predador(X)
```

Ambas compartilham condições `animal(X)` e `mamifero(X)` → Compartilham nós na rede!

#### 6.1.5 Vantagens do RETE

1. **Eficiência:** Reduz drasticamente comparações redundantes
   - Sistemas com 1000+ regras: speedup de **100-1000x**

2. **Compartilhamento:** Condições comuns entre regras são computadas uma vez

3. **Incremental:** Apenas mudanças são processadas

4. **Escalabilidade:** Viabiliza sistemas com milhares de regras

#### 6.1.6 Desvantagens

1. **Memória:** Rede RETE consome muita memória (trade-off tempo/espaço)
2. **Complexidade:** Implementação é significativamente mais complexa
3. **Compilação:** Fase inicial de compilação pode ser lenta

**Sistemas que Usam RETE:**
- CLIPS (C Language Integrated Production System)
- Drools (Java, usado em aplicações empresariais)
- Jess (Java Expert System Shell)
- SOAR (arquitetura cognitiva)

#### 6.1.7 Experimento: casamento ingênuo × indexado (a ideia do RETE)

Implementamos duas versões do *forward chaining* que derivam **exatamente os mesmos fatos**:
- **Ingênua:** a cada ciclo, testa todas as condições de todas as regras;
- **Indexada:** quando um fato novo entra, só olha as regras que **mencionam** esse fato e guarda, para cada regra, quantas condições já foram satisfeitas (*memória parcial* — o coração do RETE).

In [ ]:
def forward_ingenuo(regras, fatos):
    # Ciclo reconhecer-agir ingênuo (como a nossa classe SistemaProducao):
    # a CADA disparo, o MATCH testa de novo todas as condições de todas as regras.
    fatos, comparacoes = set(fatos), 0
    while True:
        conflito = []
        for nome, conds, concl in regras:
            comparacoes += len(conds)
            if concl not in fatos and all(c in fatos for c in conds):
                conflito.append(concl)
        if not conflito:
            return fatos, comparacoes
        fatos.add(conflito[0])          # dispara UMA regra e recomeça o ciclo


def forward_indexado(regras, fatos):
    # Ideia central do RETE: quando um fato entra, só reavaliamos as regras que o MENCIONAM,
    # e cada regra guarda quantas condições já foram satisfeitas (memória parcial).
    indice = {}
    for k, (nome, conds, concl) in enumerate(regras):
        for c in conds:
            indice.setdefault(c, []).append(k)
    faltam = [len(set(conds)) for _, conds, _ in regras]
    fatos_ok, fila, comparacoes = set(), list(fatos), 0
    while fila:
        f = fila.pop()
        if f in fatos_ok:
            continue
        fatos_ok.add(f)
        for k in indice.get(f, []):                     # só as regras afetadas pelo fato novo
            comparacoes += 1
            faltam[k] -= 1
            if faltam[k] == 0 and regras[k][2] not in fatos_ok:
                fila.append(regras[k][2])
    return fatos_ok, comparacoes


tamanhos, c_ing, c_idx = [], [], []
for n in [10, 25, 50, 100, 200, 400]:
    regras_n, ini_n = gerar_base(regras_por_nivel=n, semente=1)
    f1, a = forward_ingenuo(regras_n, ini_n)
    f2, b = forward_indexado(regras_n, ini_n)
    assert f1 == f2, 'os dois métodos devem derivar exatamente os mesmos fatos'
    tamanhos.append(len(regras_n)); c_ing.append(a); c_idx.append(b)
    print(f'{len(regras_n):5d} regras | ingênuo: {a:8d} comparações | indexado: {b:6d} | ganho ≈ {a / b:5.1f}x')

plt.figure(figsize=(8, 4))
plt.plot(tamanhos, c_ing, 'o-', color=CORES['vermelho'], label='ingênuo (testa tudo a cada ciclo)')
plt.plot(tamanhos, c_idx, 's-', color=CORES['verde'], label='indexado (estilo RETE)')
plt.xlabel('nº de regras'); plt.ylabel('comparações'); plt.yscale('log'); plt.legend()
plt.title('Mesmo resultado, custo muito diferente'); plt.grid(alpha=0.3); plt.show()

#### 🔎 Como interpretar o resultado

- **Primeiro, a garantia de correção:** o `assert` confirma que as duas versões derivam **exatamente os mesmos fatos**. A otimização muda o **custo**, nunca o **resultado**.
- **Ingênuo (vermelho):** cresce muito rápido, de 80 para **259.200** comparações. **Por quê?** No ciclo reconhecer–agir, depois de **cada** disparo o *match* testa de novo **todas** as condições de **todas** as regras. Custo ≈ (nº de disparos) × (nº de regras), e como os dois crescem juntos, o custo cresce de forma aproximadamente **quadrática**.
- **Indexado (verde):** cada fato novo só "acorda" as regras que o mencionam, e cada regra lembra quantas condições já foram satisfeitas. Custo ≈ nº de pares (fato, regra que o usa), que cresce de forma **linear**.
- **Ganho:** de ~9× (base pequena) a **~100×** (bases grandes). É exatamente o que Forgy observou: entre dois ciclos, a memória de trabalho **muda muito pouco**, então recalcular tudo é desperdício.
- **💡 Conceito-chave, o trade-off tempo × memória:** a versão indexada precisa **guardar** o índice e os contadores (`faltam`). O RETE completo guarda ainda mais (memórias *alfa* e *beta* com casamentos parciais). Ele gasta **memória** para economizar **tempo**.

### 6.2 Estratégias de Resolução de Conflitos

Quando múltiplas regras são aplicáveis (conjunto de conflitos não-vazio), sistema precisa **escolher uma**. Diferentes estratégias produzem comportamentos diferentes.

#### 6.2.1 Estratégias Clássicas

**1. Especificidade (Specificity)**

Regras **mais específicas** (mais condições) têm prioridade.

*Justificativa:* Regra específica captura situação particular; regra geral é "fallback".

**Exemplo:**
```
R1: SE animal(X) ENTÃO vertebrado(X)  [1 condição]
R2: SE animal(X) E mamifero(X) ENTÃO tem_pelos(X)  [2 condições]
```
Se ambas aplicáveis → R2 dispara primeiro (mais específica).

**2. Recência (Recency)**

Regras que usam **fatos mais recentes** têm prioridade.

*Justificativa:* Fatos recentes são mais "relevantes" no contexto atual.

**Implementação:** Cada fato tem timestamp; regra que usa fatos com maior timestamp vence.

**3. Refratariedade (Refraction)**

Regra não pode disparar **duas vezes** com mesma combinação de fatos.

*Justificativa:* Previne loops infinitos e comportamento repetitivo.

**Exemplo:**
```
SE temperatura_alta ENTÃO acionar_ventilador
```
Sem refração: dispara infinitamente (temperatura ainda alta após acionar).
Com refração: dispara uma vez; não dispara novamente.

**4. Ordem de Definição**

Regras definidas **primeiro** têm prioridade.

*Justificativa:* Simples e previsível; programador controla ordem.

**5. Prioridade Explícita**

Programador atribui **prioridade numérica** a cada regra.

*Justificativa:* Máximo controle; útil quando lógica de domínio é complexa.

**6. Aleatoriedade**

Escolhe regra **aleatoriamente** do conjunto de conflitos.

*Justificativa:* Útil para exploração, simulações, ou quando ordem não importa.

#### 6.2.2 Estratégias Compostas

Na prática, sistemas combinam múltiplas estratégias em **hierarquia**:

**Exemplo — OPS5** (estratégia **LEX**):
1. Refratariedade (elimina instanciações já disparadas)
2. Recência (prefere fatos recentes)
3. Especificidade (mais condições)
4. Escolha arbitrária entre as empatadas

O OPS5 também oferece a estratégia **MEA** (*Means-Ends Analysis*), igual à LEX, mas que dá prioridade à recência do **primeiro** elemento da condição — normalmente um fato que representa a **meta** atual. Isso facilita escrever sistemas orientados a metas.

**Exemplo — CLIPS:**
1. Prioridade explícita (salience)
2. Especificidade
3. Ordem de ativação (FIFO, LIFO, etc.)

#### 6.2.3 Impacto no Comportamento

Escolha da estratégia **afeta dramaticamente** o comportamento do sistema:

**Exemplo — Sistema de Recomendação:**

```
R1: SE cliente_novo ENTÃO oferecer_tutorial
R2: SE cliente_frequente ENTÃO oferecer_desconto
R3: SE carrinho_vazio ENTÃO sugerir_produtos
```

Com **recência:** Se cliente acabou de esvaziar carrinho → R3 dispara.
Com **especificidade:** Todas têm 1 condição → empate → usa critério secundário.
Com **prioridade explícita:** Depende do que designer considera mais importante.

#### 6.2.4 Exemplo Prático — Conflito Muito Sério

In [ ]:
# Demonstração de diferentes estratégias de resolução
import random

class SistemaComEstrategias(SistemaProducao):
    """
    Extensão do sistema básico com múltiplas estratégias de resolução.
    """

    def _select(self, conflito: List[Regra]) -> Regra:
        """
        Resolução de conflitos com estratégias avançadas.
        """
        if not conflito:
            return None

        if self.estrategia == 'prioridade':
            return max(conflito, key=lambda r: r.prioridade)

        elif self.estrategia == 'especificidade':
            return max(conflito, key=lambda r: len(r.condicoes))

        elif self.estrategia == 'aleatoria':
            return random.choice(conflito)

        elif self.estrategia == 'composta':
            # Hierarquia: prioridade > especificidade > primeira
            max_prioridade = max(r.prioridade for r in conflito)
            candidatos = [r for r in conflito if r.prioridade == max_prioridade]

            if len(candidatos) == 1:
                return candidatos[0]

            max_especificidade = max(len(r.condicoes) for r in candidatos)
            candidatos = [r for r in candidatos if len(r.condicoes) == max_especificidade]

            return candidatos[0]  # Primeira entre as empatadas

        else:  # 'primeira'
            return conflito[0]

# Exemplo: Comparando estratégias
print("="*70)
print("COMPARAÇÃO DE ESTRATÉGIAS DE RESOLUÇÃO DE CONFLITOS")
print("="*70)

for estrategia_nome in ['prioridade', 'especificidade', 'primeira']:
    print(f"\n{'='*70}")
    print(f"ESTRATÉGIA: {estrategia_nome.upper()}")
    print(f"{'='*70}")

    sistema = SistemaComEstrategias(estrategia_resolucao=estrategia_nome, debug=False)

    # Mesmas regras, mas ordem/prioridades diferentes criam conflitos
    sistema.adicionar_regra("RegA", ["X"], ["ADD A"], prioridade=1)
    sistema.adicionar_regra("RegB", ["X"], ["ADD B"], prioridade=5)
    sistema.adicionar_regra("RegC", ["X", "Y"], ["ADD C"], prioridade=3)

    sistema.adicionar_fato("X")
    sistema.adicionar_fato("Y")

    sistema.executar(max_iteracoes=10)

    print(f"\n→ Sequência de disparo: {sistema.historico_regras}")
    print(f"→ Fatos finais: {sorted(sistema.memoria_trabalho)}")

print("\n" + "="*70)
print("ANÁLISE:")
print("="*70)
print("""
• PRIORIDADE: Dispara RegB (prioridade 5), depois RegC (3), depois RegA (1)
• ESPECIFICIDADE: Dispara RegC primeiro (2 condições); RegA e RegB empatam (1 condição) e vence a primeira definida
• PRIMEIRA: Dispara na ordem de definição (RegA, RegB, RegC)

Conclusão: a estratégia determina a ORDEM dos disparos. Aqui o estado final é o mesmo,
porque nenhuma regra remove fatos — na próxima seção veremos um caso em que a DECISÃO muda.
""")

#### 🔎 Como interpretar o resultado

As três regras ficam aplicáveis **ao mesmo tempo** (todas dependem de `X`, que já está na memória). O conjunto de conflito inicial é `{RegA, RegB, RegC}`, e cada estratégia escolhe uma ordem diferente:

| Estratégia | Ordem | Critério |
|---|---|---|
| Prioridade | RegB → RegC → RegA | Prioridades 5 > 3 > 1 |
| Especificidade | RegC → RegA → RegB | RegC tem 2 condições; RegA e RegB empatam (1 condição) e vence a primeira definida |
| Primeira | RegA → RegB → RegC | Ordem em que as regras foram escritas |

- **Repare que os fatos finais são iguais** (`A, B, C, X, Y`): nenhuma regra **remove** fatos nem depende de `NOT`, então disparar uma não impede as outras. Só a **ordem** muda.
- **Quando a ordem importa?** Quando há ações que **removem** fatos, condições com **negação**, ou ações com efeitos externos (enviar um e-mail, mover um robô). É o que mostra o próximo exemplo.

#### 6.2.5 🎛️ Interativo: quando a estratégia muda a **decisão**

No exemplo anterior todas as estratégias chegavam ao mesmo estado final. Mas, quando uma regra **remove** fatos ou depende de uma **negação** (`NOT`), disparar uma regra pode **impedir** outra. Uma casa inteligente sente **calor**, tem a **janela aberta** e há **vento fresco**:

| Regra | Condições | Ações | Prioridade |
|---|---|---|:-:|
| `R_Ar` | calor ∧ ¬ventilando | liga o ar, **fecha a janela** | 5 |
| `R_Vent` | calor ∧ janela_aberta ∧ vento_fresco | ventilação natural | 1 |
| `R_Conta` | ar_ligado | conta de luz alta | 0 |

In [ ]:
# 🎛️ Casa inteligente: duas regras disputam o controle do clima.
# R_Ar  : prioridade alta, 2 condições → liga o ar e FECHA a janela
# R_Vent: prioridade baixa, 3 condições → aproveita o vento (ventilação natural)
def casa_inteligente(estrategia, mostrar=True):
    s = SistemaComEstrategias(estrategia_resolucao=estrategia, debug=False)
    s.adicionar_regra('R_Ar', ['calor', 'NOT ventilando'], ['ADD ar_ligado', 'REMOVE janela_aberta'], prioridade=5)
    s.adicionar_regra('R_Vent', ['calor', 'janela_aberta', 'vento_fresco'], ['ADD ventilando'], prioridade=1)
    s.adicionar_regra('R_Conta', ['ar_ligado'], ['ADD conta_de_luz_alta'], prioridade=0)
    for f in ['calor', 'janela_aberta', 'vento_fresco']:
        s.adicionar_fato(f)
    random.seed(3)
    executar_silencioso(s, max_iteracoes=10)
    if mostrar:
        print(f'Estratégia: {estrategia.upper()}')
        print('  Regras disparadas :', s.historico_regras)
        print('  Memória final     :', sorted(s.memoria_trabalho))
        decisao = 'ar-condicionado ❄️  (gasta energia)' if 'ar_ligado' in s.memoria_trabalho else 'ventilação natural 🌬️  (economiza)'
        print('  DECISÃO DA CASA   :', decisao)
    return s

def mostrar_casa(estrategia):
    casa_inteligente(estrategia)

interact(mostrar_casa, estrategia=widgets.Dropdown(
    options=['prioridade', 'especificidade', 'primeira', 'aleatoria'], value='prioridade', description='Estratégia'));

In [ ]:
# Comparando todas as estratégias de uma vez
for e in ['prioridade', 'especificidade', 'primeira']:
    s = casa_inteligente(e, mostrar=False)
    print(f'{e:<15} → disparos {s.historico_regras!s:<32} → {"AR LIGADO" if "ar_ligado" in s.memoria_trabalho else "VENTILAÇÃO"}')

#### 🔎 Como interpretar o resultado

| Estratégia | Quem vence o conflito inicial | Consequência | Decisão final |
|---|---|---|---|
| **Prioridade** | `R_Ar` (prioridade 5) | Fecha a janela, então `R_Vent` **perde** uma condição e nunca mais se aplica | ❄️ Ar ligado + conta alta |
| **Especificidade** | `R_Vent` (3 condições) | Cria `ventilando`, então `R_Ar` **perde** a condição `NOT ventilando` | 🌬️ Ventilação natural |
| **Primeira** | `R_Ar` (escrita primeiro) | Igual à prioridade | ❄️ Ar ligado |

- **O ponto central:** aqui a estratégia não muda só a **ordem**, mas a **decisão**. A regra que dispara primeiro **destrói as condições** da outra. Por isso a resolução de conflito é uma decisão de **projeto**, não um detalhe de implementação.
- **Qual estratégia está "certa"?** Depende do objetivo. A **especificidade** costuma acertar mais: a regra com mais condições descreve uma situação **mais particular** (calor **com** vento fresco **e** janela aberta), e por isso deve prevalecer sobre a regra genérica. É o mesmo princípio do "caso especial antes do caso geral".
- **Estratégia aleatória:** cada execução pode dar um resultado diferente. É útil em simulações e jogos (comportamento menos previsível), mas é **inaceitável** em sistemas críticos, que precisam ser **reprodutíveis e auditáveis**.

#### 6.2.6 Refratariedade e *loops* na prática

Três regras que "brigam" entre si (é o Exercício 4 da lista, agora executado):

```
R2: SE B      ENTÃO REMOVE A, REMOVE B
R1: SE A      ENTÃO ADD B
R3: SE NOT A  ENTÃO ADD A
```
(É uma variação do Exercício 4 da lista: aqui `R2` também consome `B`, o que torna o *loop* bem visível.)

In [ ]:
class SistemaSemRefracao(SistemaComEstrategias):
    # Remove a refratariedade: qualquer regra aplicável pode disparar de novo.
    def _instanciacao(self, regra):
        return ('sem-refracao', len(self.historico_regras))      # sempre "nova" → nunca é bloqueada


def montar_loop(classe):
    s = classe(estrategia_resolucao='primeira', debug=False)
    s.adicionar_regra('R2', ['B'], ['REMOVE A', 'REMOVE B'])
    s.adicionar_regra('R1', ['A'], ['ADD B'])
    s.adicionar_regra('R3', ['NOT A'], ['ADD A'])
    s.adicionar_fato('A')
    return executar_silencioso(s, max_iteracoes=16)


sem = montar_loop(SistemaSemRefracao)
com = montar_loop(SistemaComEstrategias)
print('SEM refratariedade:', sem.historico_regras, f'({len(sem.historico_regras)} disparos — parou só pelo limite)')
print('COM refratariedade:', com.historico_regras, f'({len(com.historico_regras)} disparos — parou sozinho)')

fig, axs = plt.subplots(2, 1, figsize=(12, 4))
for ax, s, titulo in [(axs[0], sem, 'SEM refratariedade: A liga e desliga para sempre (loop)'),
                      (axs[1], com, 'COM refratariedade: o sistema estabiliza')]:
    estados = [t['memoria'] for t in s.trace]
    M = np.array([[1 if f in m else 0 for m in estados] for f in ['A', 'B']])
    ax.imshow(M, cmap='Blues', aspect='auto', vmin=0, vmax=1.3)
    ax.set_yticks([0, 1]); ax.set_yticklabels(['A', 'B'])
    ax.set_xticks(range(len(estados)))
    ax.set_xticklabels([t['escolhida'] or 'FIM' for t in s.trace], fontsize=8)
    ax.set_title(titulo, fontsize=10)
plt.tight_layout(); plt.show()

#### 🔎 Como interpretar o resultado

- **Mapa de calor:** cada coluna é um ciclo (com o nome da regra disparada embaixo); azul escuro = fato **presente**.
- **SEM refratariedade (em cima):** o padrão `R1 → R2 → R3` se repete **indefinidamente**: A cria B, B apaga A e B, a ausência de A recria A... O sistema só parou porque impusemos um limite de 16 ciclos. Em produção, isso seria um programa **travado**.
- **COM refratariedade (embaixo):** depois de `R1, R2, R3, R1, R2`, a regra **R3** não pode disparar de novo, porque sua instanciação (a regra **R3** sem nenhum fato positivo) **já foi usada**. O sistema **estabiliza** sozinho em 5 disparos.
- **Por que R1 pôde disparar duas vezes mesmo com refratariedade?** Porque, da segunda vez, o fato `A` tinha sido **removido e recriado**. É um fato **novo** (outro *timestamp*), então é outra instanciação. Isso é o comportamento desejado: se o mundo mudou, a regra deve poder reagir de novo.
- **💡 Conceito-chave:** refratariedade = "não reaja duas vezes **ao mesmo estímulo**". Ela evita *loops* triviais, mas **não** garante parada em todos os casos. Projetar regras que convergem continua sendo responsabilidade do engenheiro de conhecimento.

### 6.3 Outras Otimizações e Técnicas Avançadas

#### 6.3.1 Indexação de Fatos

Usar **estruturas de dados eficientes** para armazenar fatos:
- Hash tables para busca O(1)
- Árvores de decisão para fatos estruturados
- Índices por tipo, predicado, etc.

#### 6.3.2 Compilação de Regras

Transformar regras em **código executável**:
- CLIPS compila regras para C
- Drools compila para bytecode Java
- Eliminam overhead de interpretação

#### 6.3.3 Paralelização

**OPS-R** e **RETE-II** exploram paralelismo:
- Fase MATCH pode ser paralelizada (regras independentes)
- Múltiplas regras testadas simultaneamente
- Desafio: sincronização da memória de trabalho

#### 6.3.4 Explicação e Justificação

Sistemas avançados mantêm **Truth Maintenance Systems (TMS)**:
- Rastreia **justificativas** para cada fato
- Permite explicar **por que** conclusão foi alcançada
- Suporta **retração** de fatos e propagação de mudanças

**Exemplo:**
```
Fato: paciente_tem_pneumonia
Justificativa: 
  ← Regra R3 [tem_infeccao, dificuldade_respirar, tosse_persistente]
    ← tem_infeccao justificado por Regra R1 [febre, dor_corpo]
      ← febre: observado pelo médico
      ← dor_corpo: relatado pelo paciente
```

#### 6.3.5 Incerteza e Probabilidade

**MYCIN** usava **fatores de certeza** (Certainty Factors):
```
SE febre(paciente) E dor_garganta(paciente) 
ENTÃO infeccao_bacteriana(paciente) [CF = 0.8]
```

Sistemas modernos usam **redes Bayesianas** ou **lógica fuzzy** para lidar com incerteza.

---

## 7. Aplicações Reais de Sistemas de Produção

### 7.1 Sistemas Especialistas Clássicos

#### MYCIN (1972-1980) — Stanford

**Domínio:** Diagnóstico de infecções sanguíneas e recomendação de antibióticos.

**Características:**
- ~600 regras de produção
- Fatores de certeza para lidar com incerteza
- Desempenho comparável a especialistas humanos
- Capacidade de explicar raciocínio

**Exemplo de Regra:**
```
SE:
  1) Infecção requer terapia
  2) Tipo de infecção é primária-bacteremia
  3) Suspeita de entrada da infecção é trato gastrointestinal
ENTÃO:
  Há evidência sugestiva (0.7) de que identidade do organismo
  é bacteroides
```

**Impacto:** MYCIN nunca foi usado clinicamente (questões regulatórias), mas:
- Provou viabilidade de sistemas especialistas
- Influenciou desenvolvimento de EMYCIN (shell genérico)
- Inspirou centenas de outros sistemas

#### DENDRAL (1965-1983) — Stanford

**Domínio:** Análise de espectros de massa para identificar estruturas moleculares.

**Características:**
- Primeiro sistema especialista de sucesso
- Combinava regras heurísticas com busca
- Performance de nível profissional em química orgânica

**Legado:** Publicou artigos científicos; algoritmos ainda usados em química computacional.

#### XCON / R1 (1978-1990) — DEC/CMU

**Domínio:** Configuração de sistemas computacionais VAX.

**Características:**
- ~10,000 regras
- Configurava componentes de hardware (compatibilidade, requisitos de energia, etc.)
- **Primeiro sistema especialista comercialmente bem-sucedido**

**Impacto:**
- Economizou dezenas de milhões de dólares para DEC
- Reduziu erros de configuração de 35% para <2%
- Inspirou boom de sistemas especialistas nos anos 80

### 7.2 Aplicações Industriais Modernas

#### 7.2.1 Motores de Regras de Negócio

**Drools (JBoss)** — Amplamente usado em empresas:
- **Bancos:** Detecção de fraude, aprovação de crédito
- **Seguros:** Cálculo de prêmios, avaliação de risco
- **Telecomunicações:** Validação de planos, promoções

**Exemplo — Regra de Aprovação de Crédito:**
```java
rule "Aprovar crédito de baixo risco"
when
    Cliente(score > 700, renda > 5000)
    Solicitacao(valor < renda * 3)
then
    aprovar();
    notificar("Crédito aprovado automaticamente");
end
```

#### 7.2.2 Sistemas de Controle Industrial

**CLIPS** — Usado em:
- **NASA:** Controle de missões espaciais (shuttle, ISS)
- **Indústria:** Automação, monitoramento de processos
- **Militar:** Sistemas de decisão táticos

**Exemplo — Controle de Reator:**
```clips
(defrule emergencia-pressao
   (pressao ?p&:(> ?p 100))
   (temperatura ?t&:(> ?t 300))
   =>
   (assert (alarme-critico ativado))
   (acionar-valvula-alivio))
```

#### 7.2.3 Jogos e Simulações

**IA de NPCs (Non-Player Characters):**
- Sistemas de produção controlam comportamento de agentes
- Regras para decisões táticas, reações a eventos

**Exemplo — NPC Inimigo:**
```
SE vida_baixa E tem_pocao ENTÃO usar_pocao
SE jogador_proximo E tem_municao ENTÃO atirar
SE sem_municao E jogador_longe ENTÃO buscar_municao
SE jogador_muito_forte ENTÃO fugir
```

**Jogos notáveis:**
- *The Sims* — Comportamento de personagens
- *F.E.A.R.* — IA tática avançada de inimigos
- Jogos de estratégia (Age of Empires, StarCraft) — IA de oponentes

### 7.3 Arquiteturas Cognitivas

#### SOAR (State, Operator And Result)

**Propósito:** Arquitetura cognitiva unificada para inteligência geral.

**Base:** Sistema de produção estendido com:
- Múltiplos níveis de controle
- Aprendizado por chunking (generalização)
- Resolução de impasses via subgoaling

**Aplicações:**
- Robótica (University of Michigan robots)
- Jogos (Unreal Tournament bots)
- Treinamento militar (simulações táticas)

#### ACT-R (Adaptive Control of Thought—Rational)

**Propósito:** Modelar cognição humana.

**Base:** Regras de produção + módulos cognitivos (visão, memória, motor).

**Aplicações:**
- Psicologia cognitiva (modelar experimentos)
- Design de interfaces (prever comportamento de usuários)
- Educação (tutores inteligentes)

### 7.4 Aplicações Emergentes

#### 7.4.1 Internet das Coisas (IoT)

Sistemas de produção em **edge computing**:
- Dispositivos embarcados com regras locais
- Reação rápida sem latência de nuvem

**Exemplo — Casa Inteligente:**
```
SE movimento_detectado E horario_noturno E luzes_apagadas
ENTÃO acender_luzes_nivel_baixo

SE todos_saíram E janelas_abertas E chuva_detectada
ENTÃO fechar_janelas E notificar_proprietario
```

#### 7.4.2 Compliance e Regulação

Verificação automática de conformidade:
- GDPR (privacidade de dados)
- SOX (conformidade financeira)
- Regulamentações setoriais

**Exemplo:**
```
SE dados_pessoais_coletados E sem_consentimento_explicito
ENTÃO violacao_GDPR E gerar_alerta_compliance
```

#### 7.4.3 Sistemas Híbridos (Symbolic + Neural)

**Neurosimbolic AI:** Combina regras lógicas com redes neurais.

- **Regras:** Conhecimento explícito, garantias formais
- **Redes neurais:** Aprendizado de padrões, generalização

**Exemplo — Veículos Autônomos:**
- Rede neural: Percepção (detectar pedestres, placas)
- Regras: Decisões de alto nível ("SE pedestre_atravessando ENTÃO frear")

Essa abordagem combina interpretabilidade de regras com capacidade de aprendizado de redes neurais.

### 7.5 Exemplo completo: um NPC de jogo controlado por regras

Personagens de jogos (*NPCs*) são um uso clássico de sistemas de produção **reativos**: a cada instante (*tick*), os **sensores** atualizam a memória de trabalho e o motor escolhe **uma** ação pela **prioridade**.

| Prioridade | Regra | Condição | Ação |
|:-:|---|---|---|
| 5 | Fugir | vida < 30 e sem poções | afasta-se |
| 4 | Beber poção | vida < 40 e tem poção | +50 de vida |
| 3 | Recuar | sem munição e jogador perto (≤ 4) | afasta-se |
| 3 | Recarregar | sem munição e jogador longe (> 4) | munição = 3 |
| 2 | Atirar | tem munição e jogador a ≤ 6 | dispara |
| 1 | Perseguir | jogador a > 6 | aproxima-se |
| 0 | Vigiar | sempre | fica parado |

In [ ]:
# Um NPC (personagem não jogável) controlado por um sistema de produção REATIVO.
# A cada "tick" do jogo: sensores atualizam a memória → MATCH → SELECT (prioridade) → EXECUTE.
REGRAS_NPC = [  # (prioridade, nome, condição, ação)
    (5, 'Fugir',        lambda m: m['vida'] < 30 and m['pocoes'] == 0, 'afastar'),
    (4, 'Beber poção',  lambda m: m['vida'] < 40 and m['pocoes'] > 0,  'pocao'),
    (3, 'Recuar',       lambda m: m['municao'] == 0 and m['dist'] <= 4, 'afastar'),
    (3, 'Recarregar',   lambda m: m['municao'] == 0 and m['dist'] > 4,  'recarregar'),
    (2, 'Atirar',       lambda m: m['municao'] > 0 and m['dist'] <= 6,  'atirar'),
    (1, 'Perseguir',    lambda m: m['dist'] > 6,                        'aproximar'),
    (0, 'Vigiar',       lambda m: True,                                 'parado'),
]


def simular_npc(semente=4, ticks=45):
    rng = np.random.default_rng(semente)
    npc = {'pos': 1, 'vida': 100, 'municao': 3, 'pocoes': 1}
    jogador = {'pos': 19, 'vida': 100}
    historico = []
    for t in range(ticks):
        m = dict(npc, dist=abs(jogador['pos'] - npc['pos']))          # sensores → memória de trabalho
        conflito = [r for r in REGRAS_NPC if r[2](m)]
        prio, nome, _, acao = max(conflito, key=lambda r: r[0])        # resolução de conflito: prioridade
        direcao = np.sign(jogador['pos'] - npc['pos']) or 1
        tiro_npc = tiro_jog = False
        if acao == 'aproximar':
            npc['pos'] += direcao
        elif acao == 'afastar':
            npc['pos'] = int(np.clip(npc['pos'] - 2 * direcao, 0, 20))
        elif acao == 'atirar':
            npc['municao'] -= 1; tiro_npc = True
            if rng.random() < 0.7:
                jogador['vida'] -= 15
        elif acao == 'recarregar':
            npc['municao'] = 3
        elif acao == 'pocao':
            npc['pocoes'] -= 1; npc['vida'] = min(100, npc['vida'] + 50)
        # o jogador (ambiente) também age
        dist = abs(jogador['pos'] - npc['pos'])
        if dist <= 6 and rng.random() < 0.6:
            npc['vida'] -= 12; tiro_jog = True
        jogador['pos'] = int(np.clip(jogador['pos'] + rng.choice([-1, 0, 1]), 10, 20))
        historico.append(dict(t=t, regra=nome, conflito=[r[1] for r in conflito], npc=dict(npc),
                              jog=dict(jogador), tiro_npc=tiro_npc, tiro_jog=tiro_jog))
        if npc['vida'] <= 0 or jogador['vida'] <= 0:
            break
    return historico


hist_npc = simular_npc()
from collections import Counter
print('Regras disparadas ao longo da partida:', dict(Counter(h['regra'] for h in hist_npc)))
fim = hist_npc[-1]
print(f"Fim no tick {fim['t']}: vida do NPC = {fim['npc']['vida']} | vida do jogador = {fim['jog']['vida']}")

#### 🔎 Como interpretar o resultado

- **Contagem de regras:** em 45 *ticks*, o NPC usou **6 das 7** regras (só "Vigiar" nunca venceu, porque sempre havia uma regra de prioridade maior aplicável). O comportamento é **variado** embora as regras sejam simples, porque a **memória de trabalho muda a cada tick** (posição, vida, munição).
- **A "história" da partida** (você a verá na animação):
  1. *Ticks 0–10:* **Perseguir**, já que o jogador estava longe (> 6);
  2. *Ticks 11–24:* ciclo **Atirar → Recarregar → Atirar**, com o NPC trocando tiros e perdendo vida;
  3. *Tick 21:* com vida 28 (< 40) e 1 poção, **Beber poção** (prioridade 4) vence o conflito com Atirar (prioridade 2);
  4. *Ticks 25–26:* sem munição e com o jogador perto, **Recuar**;
  5. *Ticks 28–44:* vida 18 (< 30) e **nenhuma poção**, então **Fugir**, até o fim.
- **⚠️ Uma falha de projeto revelada pela simulação:** a partir do tick 32 o NPC está **encurralado** na parede (posição 0) e continua tentando "Fugir" para sempre, mesmo com 3 balas na arma. Nenhuma regra considera "fugir é impossível". Este é o famoso **comportamento quebradiço** (*brittleness*) dos sistemas de regras: eles fazem **exatamente** o que foi escrito, nem mais nem menos.
- **💡 Exercício:** escreva uma regra de prioridade 6, `SE vida < 30 E encurralado E municao > 0 ENTÃO atirar`, crie o sensor `encurralado` (posição 0 ou 20) e simule de novo.

In [ ]:
# 🎬 ANIMAÇÃO — o NPC tomando uma decisão por tick
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.2), gridspec_kw={'height_ratios': [1.1, 1]})

def quadro_npc(k):
    h = hist_npc[k]
    ax1.clear(); ax2.clear()
    ax1.set_xlim(-1, 21); ax1.set_ylim(-1, 1); ax1.set_yticks([])
    ax1.axhline(0, color='#ccc', lw=8, zorder=0)
    xn, xj = h['npc']['pos'], h['jog']['pos']
    if h['tiro_npc']:
        ax1.annotate('', xy=(xj, 0.05), xytext=(xn, 0.05), arrowprops=dict(arrowstyle='->', color='red', lw=2))
    if h['tiro_jog']:
        ax1.annotate('', xy=(xn, -0.15), xytext=(xj, -0.15), arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    ax1.scatter([xn], [0], s=500, color=CORES['vermelho'], zorder=3, label='NPC (regras)')
    ax1.scatter([xj], [0], s=500, color=CORES['azul'], zorder=3, label='Jogador')
    ax1.legend(loc='upper left', fontsize=8, ncol=2)
    ax1.set_title(f"Tick {h['t']}  |  conflito = {h['conflito']}  →  disparou: {h['regra'].upper()}", fontsize=10)
    nomes = ['vida NPC', 'munição NPC ×20', 'poções NPC ×50', 'vida jogador']
    valores = [h['npc']['vida'], h['npc']['municao'] * 20, h['npc']['pocoes'] * 50, h['jog']['vida']]
    ax2.barh(nomes, valores, color=[CORES['vermelho'], CORES['laranja'], CORES['roxo'], CORES['azul']])
    ax2.set_xlim(0, 105); ax2.axvline(30, color='k', ls=':', lw=1); ax2.axvline(40, color='gray', ls=':', lw=1)
    ax2.set_title('Memória de trabalho (sensores) — linhas pontilhadas: limiares 30 e 40 de vida', fontsize=9)
    fig.tight_layout()

anim = animation.FuncAnimation(fig, quadro_npc, frames=len(hist_npc), interval=600)
mostrar_animacao(anim)

#### 🔎 Como interpretar a animação

- **Painel de cima, a arena:** o círculo vermelho é o NPC, o azul é o jogador. Uma seta vermelha indica que o NPC atirou; uma seta azul, que ele **foi** atingido. O título mostra o **conjunto de conflito** e a regra **vencedora** daquele tick.
- **Painel de baixo, a memória de trabalho:** as barras são os valores que os **sensores** colocam na memória. As linhas pontilhadas em **30** e **40** são os limiares das regras "Fugir" e "Beber poção". Observe que a regra muda **exatamente** quando a barra vermelha cruza uma dessas linhas.
- **O que observar:**
  - O **conjunto de conflito** quase sempre tem 2 ou 3 regras (por exemplo `['Atirar', 'Vigiar']`). A **prioridade** é que decide a ação. Sem resolução de conflito, o NPC não saberia o que fazer.
  - As decisões são **reativas**: o NPC não tem plano nem memória do passado. Ele reage ao estado **atual**. Isso o torna rápido e previsível, mas incapaz de estratégias de longo prazo, como pegar munição **antes** de entrar em combate.
- **💡 Ligação com jogos reais:** essa arquitetura (regras com prioridade, reavaliadas a cada frame) é a base de muitas IAs de jogos. Versões mais sofisticadas usam máquinas de estados, *behavior trees* e planejamento (como o GOAP do jogo *F.E.A.R.*).

---

## 8. Vantagens e Desvantagens dos Sistemas de Produção

### 8.1 Vantagens

#### 8.1.1 Modularidade e Manutenibilidade

**Vantagem:** Regras são unidades independentes.

- **Fácil adicionar conhecimento:** Nova regra não afeta regras existentes
- **Fácil modificar:** Alterar regra específica sem reescrever sistema
- **Fácil remover:** Deletar regra obsoleta sem efeitos colaterais (geralmente)

**Contraste com código procedural:**
```python
# Procedural — Entrelaçado, difícil de modificar
def diagnosticar(paciente):
    if paciente.febre and paciente.tosse:
        if paciente.dor_garganta:
            return "gripe"
        elif paciente.dor_peito:
            return "pneumonia"
    # Adicionar novo diagnóstico requer modificar estrutura

# Sistema de produção — Modular
adicionar_regra("Nova_Doença", ["febre", "sintoma_X"], ["doença_Y"])
# Não modifica regras existentes!
```

#### 8.1.2 Transparência e Explicabilidade

**Vantagem:** Raciocínio é facilmente rastreável.

- **Traço de execução:** Sequência de regras disparadas
- **Justificação:** Por que conclusão foi alcançada
- **Auditoria:** Fácil verificar se sistema seguiu políticas

**Essencial em:**
- Medicina (explicar diagnósticos)
- Finanças (justificar decisões de crédito)
- Legal (compliance com regulações)

**Contraste com ML:**
- Rede neural: "Caixa preta" — difícil explicar decisões
- Sistema de produção: "Caixa de vidro" — totalmente transparente

#### 8.1.3 Naturalidade

**Vantagem:** Regras refletem forma natural de expressar conhecimento.

Especialistas humanos frequentemente expressam conhecimento como:
- *"Se o paciente tem febre alta e dor de cabeça, então considere meningite."*
- *"Se cliente é VIP, então ofereça desconto."*

Traduzir para regras IF-THEN é direto.

**Aquisição de conhecimento:** Especialistas podem revisar e validar regras facilmente.

#### 8.1.4 Separação Conhecimento-Controle

**Vantagem:** Conhecimento (regras) é separado do mecanismo de controle (motor de inferência).

- Alterar estratégia de busca sem modificar regras
- Reutilizar motor de inferência em diferentes domínios
- Otimizar controle independentemente

**Exemplo:** Mesmo conjunto de regras pode usar:
- Forward chaining (para derivar tudo)
- Backward chaining (para consultas)
- Estratégia híbrida

#### 8.1.5 Facilidade de Prototipagem

**Vantagem:** Sistemas podem ser construídos incrementalmente.

- Começar com regras básicas
- Adicionar refinamentos gradualmente
- Testar a cada adição

**Desenvolvimento ágil:** Especialista e desenvolvedor colaboram iterativamente.

### 8.2 Desvantagens

#### 8.2.1 Opacidade Direcional

**Desvantagem:** Ordem de execução não é explícita.

- **Difícil prever:** Qual regra dispara em qual ordem
- **Debugging complexo:** Comportamento emergente pode ser inesperado
- **Interações não-óbvias:** Regras podem interagir de formas sutis

**Exemplo problemático:**
```
R1: SE A ENTÃO ADD B
R2: SE B ENTÃO REMOVE A
R3: SE NOT A ENTÃO ADD C
```
Ordem de disparo determina resultado final — não é óbvio!

#### 8.2.2 Ineficiência Computacional

**Desvantagem:** Matching ingênuo é caro.

- Sem otimizações (RETE): O(regras × fatos × condições) por ciclo
- Grandes bases de regras podem ser lentas
- Problema de escalabilidade

**Mitigação:** RETE e outras otimizações ajudam, mas aumentam complexidade.

#### 8.2.3 Dificuldade com Incerteza

**Desvantagem:** Regras clássicas são **binárias** (verdadeiro/falso).

- Mundo real é incerto, probabilístico, fuzzy
- Fatores de certeza (MYCIN) são ad-hoc
- Combinação de incertezas é problemática

**Alternativas melhores para incerteza:**
- Redes Bayesianas
- Lógica Fuzzy
- Modelos probabilísticos (HMMs, CRFs)

#### 8.2.4 Aquisição de Conhecimento (Knowledge Bottleneck)

**Desvantagem:** Extrair conhecimento de especialistas é difícil e demorado.

- Especialistas nem sempre conseguem articular conhecimento
- Conhecimento pode estar implícito, tácito
- Processo de elicitação é caro

**Citação famosa — Edward Feigenbaum:**
> "The knowledge acquisition bottleneck is the major impediment to building expert systems."

**Solução moderna:** Aprendizado de máquina — aprender regras/padrões de dados.

#### 8.2.5 Manutenção de Grandes Bases

**Desvantagem:** Bases com milhares de regras se tornam difíceis de manter.

- **Inconsistências:** Regras contraditórias podem ser introduzidas
- **Redundâncias:** Múltiplas regras fazendo coisas similares
- **Cobertura:** Difícil garantir que todos os casos são cobertos

**Ferramentas necessárias:**
- Verificação de consistência
- Análise de cobertura
- Ferramentas de visualização

#### 8.2.6 Limitações Representacionais

**Desvantagem:** Nem todo conhecimento se expressa bem como regras.

- **Conhecimento temporal:** Raciocínio sobre tempo é complexo
- **Conhecimento espacial:** Relações espaciais são difíceis
- **Aprendizado:** Sistemas clássicos não aprendem automaticamente
- **Senso comum:** Difícil codificar conhecimento do dia-a-dia

**Domínios problemáticos:**
- Visão computacional (redes neurais são melhores)
- Processamento de linguagem natural (modelos estatísticos)
- Reconhecimento de padrões (ML supervisionado)

#### 8.2.7 Falha Catastrófica

**Desvantagem:** Fora do domínio de conhecimento, falha completamente.

- Sistema não generaliza além das regras codificadas
- Situações imprevistas não são tratadas
- "Brittle" — quebradiço

**Contraste com humanos:**
- Humanos usam raciocínio por analogia, senso comum
- Sistemas de produção não têm esses mecanismos (a menos que codificados explicitamente)

### 8.3 Quando Usar Sistemas de Produção?

**Use quando:**
✓ Conhecimento pode ser expresso como regras claras  
✓ Explicabilidade é crucial  
✓ Domínio é bem definido e estável  
✓ Especialistas humanos disponíveis para validar  
✓ Dados são escassos (não pode usar ML supervisionado)  
✓ Precisa de garantias formais (segurança, compliance)  

**Evite quando:**
✗ Conhecimento é difícil de articular explicitamente  
✗ Domínio é vago, complexo, ou mal definido  
✗ Dados abundantes estão disponíveis (ML é melhor)  
✗ Padrões são sutis, perceptuais (visão, fala)  
✗ Incerteza é central (use métodos probabilísticos)  
✗ Ambiente é altamente dinâmico e imprevisível  

---

## 9. Exercícios Propostos

### Exercício 1 — Sistema de Recomendação de Filmes (Nível Básico)

Implemente um sistema de produção com **forward chaining** que recomenda filmes baseado em preferências do usuário.

**Especificação:**

- **Fatos iniciais:** Gêneros preferidos (ex: `"gosta_acao"`, `"gosta_comedia"`, `"gosta_scifi"`)
- **Fatos adicionais:** Características (ex: `"prefere_filmes_longos"`, `"gosta_finais_felizes"`)

**Regras sugeridas:**
```
SE gosta_acao E gosta_scifi ENTÃO recomendar("Matrix")
SE gosta_comedia E gosta_finais_felizes ENTÃO recomendar("De Repente 30")
SE gosta_acao E NOT gosta_violencia ENTÃO recomendar("Indiana Jones")
SE gosta_scifi E prefere_filmes_longos ENTÃO recomendar("Interestelar")
```

**Tarefas:**
1. Crie pelo menos 8 regras de recomendação
2. Teste com diferentes perfis de usuário
3. Exiba o traço de execução mostrando por que cada filme foi recomendado

---

### Exercício 2 — Diagnóstico Automotivo com Backward Chaining (Nível Intermediário)

Implemente um sistema de diagnóstico para problemas automotivos usando **backward chaining**.

**Cenário:** Mecânico observa sintomas e sistema tenta provar qual é o problema.

**Sintomas possíveis:**
- `"motor_nao_liga"`, `"barulho_estranho"`, `"luz_bateria_acesa"`, `"dificuldade_arrancar"`, `"fumaca_escape"`, `"superaquecimento"`

**Problemas possíveis:**
- `"bateria_descarregada"`, `"alternador_quebrado"`, `"motor_fundido"`, `"problema_combustivel"`

**Regras de diagnóstico:**
```
SE motor_nao_liga E luz_bateria_acesa ENTÃO bateria_descarregada
SE bateria_descarregada E carro_usado_recentemente ENTÃO alternador_quebrado
SE superaquecimento E fumaca_escape ENTÃO motor_fundido
```

**Tarefas:**
1. Crie pelo menos 10 regras diagnósticas
2. Implemente sistema com backward chaining
3. Teste tentando provar diferentes problemas
4. Sistema deve exibir caminho de raciocínio

---

### Exercício 3 — Comparação de Estratégias de Resolução (Nível Intermediário)

Crie um sistema de produção que demonstra o **impacto de diferentes estratégias de resolução de conflitos**.

**Especificação:**

1. Defina um cenário onde múltiplas regras são aplicáveis simultaneamente (ex: sistema de controle de semáforos)

2. Implemente as seguintes estratégias:
   - **Prioridade explícita**
   - **Especificidade** (número de condições)
   - **Aleatória**

3. Execute o mesmo cenário com cada estratégia

4. Compare:
   - Ordem de disparo das regras
   - Estado final da memória de trabalho
   - Número de ciclos até parada

**Exemplo de cenário — Controle de Tráfego:**
```
Fatos: fluxo_norte_alto, fluxo_leste_medio, horario_pico

R1: SE fluxo_norte_alto ENTÃO priorizar_norte [prioridade=5, 1 condição]
R2: SE horario_pico E fluxo_norte_alto ENTÃO estender_verde_norte [prioridade=3, 2 condições]
R3: SE fluxo_leste_medio ENTÃO abrir_leste [prioridade=2, 1 condição]
```

**Análise:**  
Escreva um parágrafo analisando como a escolha da estratégia afeta o comportamento do sistema.

---

### Exercício 4 — Sistema com Refratariedade e Loops (Nível Avançado)

Implemente um sistema que **demonstra a importância da refratariedade**.

**Parte A — Sem Refratariedade:**

1. Crie um sistema simples **sem** mecanismo de refração
2. Adicione regras que podem causar loop infinito:
```
R1: SE A ENTÃO ADD B
R2: SE B ENTÃO REMOVE A
R3: SE NOT A ENTÃO ADD A
```
3. Execute e observe o loop

**Parte B — Com Refratariedade:**

1. Adicione mecanismo de refração
2. Execute novamente e observe que sistema para
3. Documente diferença no comportamento

**Parte C — Análise:**

Responda:
- Por que refratariedade previne loops?
- Existem situações onde refração pode ser problemática?
- Como sistemas reais (CLIPS, Drools) lidam com isso?

---

### Exercício 5 — Sistema Híbrido Forward + Backward (Nível Avançado)

Implemente um **sistema híbrido** que combina forward e backward chaining.

**Cenário:** Sistema de suporte a decisão médica.

**Estratégia:**

1. **Fase 1 — Forward Chaining:**
   - Parte de sintomas observados
   - Deriva categorias gerais (ex: `"tem_infeccao"`, `"problema_respiratorio"`)

2. **Fase 2 — Backward Chaining:**
   - Médico propõe diagnóstico específico (ex: `"pneumonia"`)
   - Sistema tenta provar usando fatos derivados na Fase 1

**Especificação técnica:**

```python
class SistemaHibrido:
    def __init__(self):
        self.forward_engine = SistemaProducao()
        self.backward_engine = SistemaBackwardChaining()
    
    def processar(self, sintomas_iniciais, objetivo):
        # Fase 1: Forward chaining para derivar fatos gerais
        for sintoma in sintomas_iniciais:
            self.forward_engine.adicionar_fato(sintoma)
        self.forward_engine.executar()
        
        # Transfere fatos derivados para backward engine
        for fato in self.forward_engine.memoria_trabalho:
            self.backward_engine.adicionar_fato(fato)
        
        # Fase 2: Backward chaining para provar objetivo
        return self.backward_engine.consultar(objetivo)
```

**Tarefas:**
1. Implemente o sistema híbrido completo
2. Crie base de conhecimento com:
   - Regras de categorização (para forward)
   - Regras de diagnóstico (para backward)
3. Teste com pelo menos 3 cenários diferentes
4. Compare com usar apenas forward ou apenas backward

**Análise final:**  
Escreva 2-3 parágrafos explicando:
- Vantagens do sistema híbrido
- Quando é apropriado usar essa abordagem
- Possíveis melhorias na implementação

In [ ]:
# Espaço para resolver os exercícios
# Exercício 1: Sistema de Recomendação de Filmes

# Seu código aqui:


In [ ]:
# Exercício 2: Diagnóstico Automotivo

# Seu código aqui:


In [ ]:
# Exercício 3: Comparação de Estratégias

# Seu código aqui:


In [ ]:
# Exercício 4: Refratariedade e Loops

# Seu código aqui:


In [ ]:
# Exercício 5: Sistema Híbrido

# Seu código aqui:


---

## 10. Referências Bibliográficas

### Livros Clássicos

1. **Newell, A. & Simon, H. A.** (1972). *Human Problem Solving*. Prentice-Hall. — Trabalho seminal sobre sistemas de produção e arquiteturas cognitivas.

2. **Russell, S. & Norvig, P.** (2020). *Artificial Intelligence: A Modern Approach* (4ª ed.). Pearson. — Capítulos 9 e 10 cobrem sistemas baseados em regras e inferência lógica.

3. **Nilsson, N. J.** (1998). *Artificial Intelligence: A New Synthesis*. Morgan Kaufmann. — Tratamento detalhado de sistemas de produção e suas aplicações.

4. **Giarratano, J. & Riley, G.** (2005). *Expert Systems: Principles and Programming* (4ª ed.). Thomson. — Guia prático com foco em CLIPS e desenvolvimento de sistemas especialistas.

5. **Luger, G. F.** (2009). *Artificial Intelligence: Structures and Strategies for Complex Problem Solving* (6ª ed.). Addison-Wesley. — Capítulo 5 sobre sistemas de produção.

### Artigos Fundamentais

6. **Forgy, C. L.** (1982). *Rete: A Fast Algorithm for the Many Pattern/Many Object Pattern Match Problem*. Artificial Intelligence, 19(1), 17-37. — Artigo original do algoritmo RETE.

7. **Newell, A.** (1990). *Unified Theories of Cognition*. Harvard University Press. — Propõe SOAR como arquitetura unificada para cognição.

8. **Buchanan, B. G. & Shortliffe, E. H.** (1984). *Rule-Based Expert Systems: The MYCIN Experiments of the Stanford Heuristic Programming Project*. Addison-Wesley. — Documentação completa do sistema MYCIN.

9. **Anderson, J. R.** (1983). *The Architecture of Cognition*. Harvard University Press. — Apresenta arquitetura ACT* (predecessora de ACT-R).

10. **Brownston, L., Farrell, R., Kant, E., & Martin, N.** (1985). *Programming Expert Systems in OPS5: An Introduction to Rule-Based Programming*. Addison-Wesley. — Manual clássico sobre OPS5.

### Recursos Online e Ferramentas

11. **CLIPS Official Website:** http://www.clipsrules.net/ — Documentação, tutoriais e downloads do CLIPS.

12. **Drools Documentation:** https://www.drools.org/ — Motor de regras de negócio em Java, amplamente usado na indústria.

13. **SOAR Cognitive Architecture:** https://soar.eecs.umich.edu/ — Recursos sobre arquitetura cognitiva SOAR.

14. **ACT-R Official Website:** http://act-r.psy.cmu.edu/ — Documentação e tutoriais sobre ACT-R.

### Artigos Históricos

15. **Post, E.** (1943). *Formal Reductions of the General Combinatorial Decision Problem*. American Journal of Mathematics, 65(2), 197-215. — Sistemas de Post, precursores teóricos.

16. **Newell, A., Shaw, J. C., & Simon, H. A.** (1958). *Elements of a Theory of Human Problem Solving*. Psychological Review, 65(3), 151-166. — GPS (General Problem Solver).

17. **Feigenbaum, E. A.** (1977). *The Art of Artificial Intelligence: Themes and Case Studies of Knowledge Engineering*. IJCAI-77. — Discussão sobre engenharia de conhecimento.

### Aplicações e Estudos de Caso

18. **McDermott, J.** (1982). *R1: A Rule-Based Configurer of Computer Systems*. Artificial Intelligence, 19(1), 39-88. — Descrição detalhada do sistema XCON/R1.

19. **Lindsay, R. K., Buchanan, B. G., Feigenbaum, E. A., & Lederberg, J.** (1993). *DENDRAL: A Case Study of the First Expert System for Scientific Hypothesis Formation*. Artificial Intelligence, 61(2), 209-261.

20. **Laird, J. E., Newell, A., & Rosenbloom, P. S.** (1987). *SOAR: An Architecture for General Intelligence*. Artificial Intelligence, 33(1), 1-64.

---

**Notebook preparado para a disciplina de Inteligência Artificial.**  
**Curso de Ciência da Computação.**